ass

crear adm_obj_tg.RhfEfectivaN RHF VARCHAR(3),[SOLO FH1]INT,[SOLO FH2]INT,[SOLO FH3]INT,[DOBLE FH]INT,[TRIPLE FH] INT
crear adm_obj_tg.NumxDnixEfectivaN |NUMDOCUMENTO as DNI,count(*) as numxdni

crear adm_obj_tg.FunnelEfectivaN -->
    tabla
        [ADM_OBJ_TG].[synBaseMaestraEfectivaNegocios]
        [ADM_OBJ_TG].[synTmpLLamadasEfectivaN]
        adm_obj_tg.NumxDnixEfectivaN
        [ADM_OBJ_TG].[synUsuariosTg]
        adm_obj_tg.RhfEfectivaN
    where a.NUMDOCUMENTO IS NOT NULL

actualizo adm_obj_tg.RhfEfectivaN 
    columnas 
        TIPO_GESTION  ='CONTACTO EFECTIVO', 
        GESTION ='EFECTIVO', 
        SUBGESTION ='VOLVER A LLAMAR'
    tabla 
        [ADM_OBJ_TG].[synVentasEfectivaN] left
    where b.dni is null and a.SUBGESTION='SI QUIERE'

actualizo adm_obj_tg.RhfEfectivaN 
    columnas 
        TIPO_GESTION  ='CONTACTO EFECTIVO', 
        GESTION ='EFECTIVO', 
        SUBGESTION ='VOLVER A LLAMAR'
    tabla [ADM_OBJ_TG].[synVentasEfectivaN] inner

actualizo adm_obj_tg.RhfEfectivaN 
    columnas Ejecutivo tNombreCompleto
    tabla [ADM_OBJ_TG].[synVentasEfectivaN] inner
          [ADM_OBJ_TG].[synUsuariosTg] inner

actualizo adm_obj_tg.RhfEfectivaN 
    columnas MntOferta = monto
    tabla [ADM_OBJ_TG].[synVentasEfectivaN] inner

actualizo adm_obj_tg.RhfEfectivaN 
    columnas Estado = isnull(replace(b.ESTADO,'-','EN PROCESO'),'EN PROCESO')
    tabla [ADM_OBJ_TG].[synVentasEfectivaN] inner

actualizo adm_obj_tg.RhfEfectivaN 
    columnas CntEstado=1
    tabla [ADM_OBJ_TG].[synVentasEfectivaN] inner
    where ESTADO='1.-VALIDADA'

actualizo adm_obj_tg.RhfEfectivaN 
    columnas FECHA_LLAMADA =convert(varchar,CONVERT(date,b.FECHA))
    , a.DIA=substring(convert(varchar,b.FECHA),9,2)
    tabla [ADM_OBJ_TG].[synVentasEfectivaN] inner

actualizo adm_obj_tg.RhfEfectivaN 
    columnas a.[NUM_DIA_HABIL]=b.[NUM_DIA_HABIL]
    a.Semana_Mes=b.Semana_Mes 
    tabla [ADM_OBJ_TG].[synDiaHabil] inner

crear adm_obj_tg.LlamadasEfectivaN1
    group by LlamadasEfectivaN1
        columnas DNI AS DNI_LLAMADAS
        , COUNT(*) AS CNT_LLAMADAS1 
    tabla [ADM_OBJ_TG].[synTmpLlamadasGeneralEfectivaN] 


crear adm_obj_tg.tDesembolso_efeneg 
    tabla
        [ADM_OBJ_TG].[DesembolsoNegociosEfe] 
    where DATENAME(MONTH,FECHA_DESEMBOLSO)=DATENAME(MONTH,GETDATE()) and YEAR(FECHA_DESEMBOLSO)=YEAR(getdate());

actualizar adm_obj_tg.FunnelEfectivaN
    columna        
        a.tMontoDesem =b.MONTO_NETO
    tabla 
        tDesembolso_efeneg

eliminar info del mes actual [ADM_OBJ_TG].[tGestionMesEfectivaN]
    tabla
        adm_obj_tg.FunnelEfectivaN
        adm_obj_tg.LlamadasEfectivaN1


-----------------------------------------------
tambien hay q preparar el resumen


In [ ]:
select a.*  from (
    select 
        tMesGestion,NUMERO_DOCUMENTO+' '+convert(varchar,fecha_envio) llave,'Entregados' MedicionDatos,sum(NRG) Q,AÑO_DURACION_BASE 
    from [ADM_OBJ_TG].[tGestionMesEfectivaN] 
    where NRG=1 
    GROUP BY NUMERO_DOCUMENTO,FECHA_ENVIO,tMesGestion,AÑO_DURACION_BASE
    UNION ALL
    select 
        tMesGestion,NUMERO_DOCUMENTO+' '+convert(varchar,fecha_envio) llave,'Recorridos' MedicionDatos,sum(RECORRIDO) Q,AÑO_DURACION_BASE 
    from [ADM_OBJ_TG].[tGestionMesEfectivaN] 
    where RECORRIDO=1 
    GROUP BY NUMERO_DOCUMENTO,FECHA_ENVIO,tMesGestion,AÑO_DURACION_BASE
    UNION ALL
    select 
        tMesGestion,NUMERO_DOCUMENTO+' '+convert(varchar,fecha_envio) llave,'Llamados' MedicionDatos,sum(CNT_LLAMADAS) Q,AÑO_DURACION_BASE 
    from [ADM_OBJ_TG].[tGestionMesEfectivaN] 
    where RECORRIDO=1 
    GROUP BY NUMERO_DOCUMENTO,FECHA_ENVIO,tMesGestion,AÑO_DURACION_BASE
    UNION ALL
    select 
        tMesGestion,NUMERO_DOCUMENTO+' '+convert(varchar,fecha_envio) llave,'Titular' MedicionDatos,sum(CET) Q,AÑO_DURACION_BASE 
    from [ADM_OBJ_TG].[tGestionMesEfectivaN] 
    where CET=1 
    GROUP BY NUMERO_DOCUMENTO,FECHA_ENVIO,tMesGestion,AÑO_DURACION_BASE
    UNION ALL
    select 
        tMesGestion,NUMERO_DOCUMENTO+' '+convert(varchar,fecha_envio) llave,'Ventas' MedicionDatos,sum(CNTVTAS) Q,AÑO_DURACION_BASE 
    from [ADM_OBJ_TG].[tGestionMesEfectivaN] 
    where CNTVTAS=1 
    GROUP BY NUMERO_DOCUMENTO,FECHA_ENVIO,tMesGestion,AÑO_DURACION_BASE 
) a where tMesGestion=DATENAME(MONTH,GETDATE()) and AÑO_DURACION_BASE='2025';


select vnt.*,
cast(dt.NUM_DIA_HABIL as int) dia_,dt.[Semana_Mes],tp.[tSupervisor]
into [ADM_OBJ_TG].VentasEfectivaNFunnel
from [ADM_OBJ_TG].[synVentasEfectivaNFunnel] VNT 
INNER JOIN [ADM_OBJ_TG].[synDiaHabil] DT ON VNT.FECHA=DT.Fecha
inner join [ADM_OBJ_TG].[synUsuariosTg] tp on vnt.DNIEjecutivo=tp.tDocumentoVici
WHERE DATENAME(MONTH,VNT.FECHA) in (@MES1,@MES2,@MES3)
and CAMPANA='Negocios'
order by FECHA


In [ ]:
ssss

In [1]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from unidecode import unidecode
from sqlalchemy import create_engine
from sqlalchemy import text

fecha_mes_base='2026-07-01'
tipi_cond1='RECLUTAMIENTO'
servidor_01=64

campana='negocios'

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_zeus
db_sql = "ODIN"
user_sql = user_zeus
pwd_sql = pwd_zeus
engine_zeus = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_sa
db_sql = "ODIN"
user_sql = user_sa
pwd_sql = pwd_sa
engine_sa = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_zeus
db_sql = "SAMANTHA"
user_sql = user_zeus
pwd_sql = pwd_zeus
engine_samantha = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

def nombre_mes_anio(fecha_mes_base):
    from datetime import datetime

    fecha = datetime.strptime(fecha_mes_base, "%Y-%m-%d")

    meses = [
        "enero", "febrero", "marzo", "abril", "mayo", "junio",
        "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"
    ]

    return f"{meses[fecha.month - 1]} {fecha.year}"


In [23]:
tb_general=set(['NUMERO_DOCUMENTO','NRG','FECHA_ENVIO','Dni','TIPO_GESTION','GESTION','SUBGESTION','SERVICIO','AÑO_DURACION_BASE','MES_DURACION_BASE','RETIRO','TIPO','LINEA_EfectivaN','RECORRIDO','CET','CONT_GEN','REGION','RANGO_OFERTA','RANGO_TASA','EDAD','Hora_Llamada','Prioridad','MARCA','MARCA2','PERFIL','Mejor_Telefono','TELEFONO','SOLO FH1','SOLO FH2','SOLO FH3','DOBLE FH','TRIPLE FH','FECHA_LLAMADA','DIA','Nombre_Campana','DESCRIPCION','DESCRIPCION2','Ejecutivo','NUMXDNI','SUPERVISOR','FLG_REP_MES_ANT','FLG_REP_MESES_ANT','tMesGestion','FLG_SIN_ENR','PROVINCIA','segundos','VAL','Numero_Campana','AGENDADOS','Estado','CntEstado','MntOferta','CNTVTAS','NUM_DIA_HABIL','Semana_Mes','llave','tMontoDesem','CNT_LLAMADAS','AREAEFECTINEGOCIOS','REP1','REP2','REP3','CONJUNTO3MESES','SEGMICROPEQUENACOMERCIAL','TIPOBASEMICROPEQUENA','Proceso','Zona','DEPARTAMENTO','LOTE','TIPO_DEUDA','MONTOREF'])

df_funel_debe_ir=set([
'NUMERO_DOCUMENTO','NRG','Dni','TIPO_GESTION','GESTION','SUBGESTION','TIPO','RECORRIDO',
'CET','CONT_GEN','Hora_Llamada','Mejor_Telefono','TELEFONO','FECHA_LLAMADA','DIA',
'Ejecutivo','SUPERVISOR','segundos','VAL','AGENDADOS','SOLO FH1','SOLO FH2',
'SOLO FH3','DOBLE FH','TRIPLE FH','NUMXDNI','Estado','CntEstado','MntOferta',
'CNTVTAS','NUM_DIA_HABIL','Semana_Mes','tMontoDesem','CNT_LLAMADAS','AREAEFECTINEGOCIOS',    
'AÑO_DURACION_BASE','MES_DURACION_BASE','FECHA_ENVIO','SERVICIO','RETIRO',
'REP1','REP2','CONJUNTO3MESES','FLG_REP_MES_ANT','FLG_REP_MESES_ANT','FLG_SIN_ENR',
'tMesGestion','DESCRIPCION','DESCRIPCION2','llave','REGION','EDAD',
'RANGO_OFERTA','TIPO_DEUDA','MONTOREF','LOTE','CONJUNTO3MESES','SEGMICROPEQUENACOMERCIAL',
'TIPOBASEMICROPEQUENA','Proceso','Zona','DEPARTAMENTO', 
    ])


df_funel=set([
'NUMERO_DOCUMENTO','NRG','Dni','TIPO_GESTION','GESTION','SUBGESTION','TIPO','RECORRIDO','CET','CONT_GEN','Hora_Llamada','Mejor_Telefono','TELEFONO','FECHA_LLAMADA','DIA','Ejecutivo','SUPERVISOR','segundos','VAL','AGENDADOS','SOLO FH1','SOLO FH2','SOLO FH3','DOBLE FH','TRIPLE FH','NUMXDNI','Estado','CntEstado','MntOferta','CNTVTAS','NUM_DIA_HABIL','Semana_Mes','tMontoDesem'    
    ])
base_real=set(['NUMDOCUMENTO','SEGMICROPEQUENACOMERCIAL','IMPDHM','TIPOBASEMICROPEQUENA','Nacimiento','Direccion','DEPARTAMENTO','PROVINCIA','DISTRITO','AREAEFECTINEGOCIOS','ruc','num_cuotas_pagadas','num_cuotas_impagas','prioridad','PERFILINICIAL','ULTIMOMONTODESEMBOLSADO','ULTIMOMONTODESEMBOLSADOPLUS20','empresa1','RangoSaldo1','empresa2','RangoSaldo2','empresa3','RangoSaldo3','flag_recurrencia_efectinegocio','CANAL_ASIGNADO','FECHA_ENVIO','MES_DURACION_BASE','AÑO_DURACION_BASE','SERVICIO','RETIRO','FLAT1','FLAT2','FLAT3','REP1','REP2','DEVUELTO','recompra','Proceso','Zona','Montos_Referenciales','REP3','REP4','MONTOREFERENCIALFT','MONTOREFERENCIALHS','flag_consentimiento','PERFIL_IC','DEUDAMYPESINEFE','DEUDAELECTRO','DEUDAMOTOS','DEUDAEFECTIVO'])

In [10]:
query = f"""
select * from MAEBA.[ADM_OBJ_TG].[tGestionMesEfectiva]
"""
df_ges=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
query = f"""
select * from ODIN.dbo.Base_Maestra_Efectiva_Vigente
"""
df_base=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)
print(df_ges.columns)
print(df_base.columns)

['NUMERO_DOCUMENTO', 'NRG', 'FECHA_ENVIO', 'Dni', 'TIPO_GESTION', 'GESTION', 'SUBGESTION', 'SERVICIO', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'RETIRO', 'TIPO', 'LINEA_Efectiva', 'RECORRIDO', 'CET', 'CONT_GEN', 'REGION', 'RANGO_OFERTA', 'RANGO_TASA', 'EDAD', 'Hora_Llamada', 'Prioridad', 'MARCA', 'MARCA2', 'PERFIL', 'Mejor_Telefono', 'TELEFONO', 'SOLO FH1', 'SOLO FH2', 'SOLO FH3', 'DOBLE FH', 'TRIPLE FH', 'FECHA_LLAMADA', 'DIA', 'Nombre_Campana', 'DESCRIPCION', 'DESCRIPCION2', 'Ejecutivo', 'NUMXDNI', 'SUPERVISOR', 'FLG_REP_MES_ANT', 'FLG_REP_MESES_ANT', 'tMesGestion', 'FLG_SIN_ENR', 'PROVINCIA', 'segundos', 'VAL', 'Numero_Campana', 'AGENDADOS', 'Estado', 'CntEstado', 'MntOferta', 'CNTVTAS', 'NUM_DIA_HABIL', 'Semana_Mes', 'llave', 'Desembolso', 'CNT_LLAMADAS', 'RHFC', 'SEGMENTO', 'NOMCOMERCIAL', 'ZONA', 'DEPARTAMENTO', 'AGENCIA', 'RANGO_HS_RS', 'RANGO_HS_PLUS', 'SITUACIONLABORAL', 'TIPOINGRESO', 'FLGCONVENIO', 'FLGSUBPROCESO_HS', 'OFERTAREF', 'LOTE']
['DNI', 'CLIENTE', 'ASIGNACION', 'C

In [11]:
tb_general=set(['NUMERO_DOCUMENTO', 'NRG', 'FECHA_ENVIO', 'Dni', 'TIPO_GESTION', 'GESTION', 'SUBGESTION', 'SERVICIO', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'RETIRO', 'TIPO', 'LINEA_Efectiva', 'RECORRIDO', 'CET', 'CONT_GEN', 'REGION', 'RANGO_OFERTA', 'RANGO_TASA', 'EDAD', 'Hora_Llamada', 'Prioridad', 'MARCA', 'MARCA2', 'PERFIL', 'Mejor_Telefono', 'TELEFONO', 'SOLO FH1', 'SOLO FH2', 'SOLO FH3', 'DOBLE FH', 'TRIPLE FH', 'FECHA_LLAMADA', 'DIA', 'Nombre_Campana', 'DESCRIPCION', 'DESCRIPCION2', 'Ejecutivo', 'NUMXDNI', 'SUPERVISOR', 'FLG_REP_MES_ANT', 'FLG_REP_MESES_ANT', 'tMesGestion', 'FLG_SIN_ENR', 'PROVINCIA', 'segundos', 'VAL', 'Numero_Campana', 'AGENDADOS', 'Estado', 'CntEstado', 'MntOferta', 'CNTVTAS', 'NUM_DIA_HABIL', 'Semana_Mes', 'llave', 'Desembolso', 'CNT_LLAMADAS', 'RHFC', 'SEGMENTO', 'NOMCOMERCIAL', 'ZONA', 'DEPARTAMENTO', 'AGENCIA', 'RANGO_HS_RS', 'RANGO_HS_PLUS', 'SITUACIONLABORAL', 'TIPOINGRESO', 'FLGCONVENIO', 'FLGSUBPROCESO_HS', 'OFERTAREF', 'LOTE']
)

df_funel_debe_ir=set([
'NUMERO_DOCUMENTO','NRG','Dni','TIPO_GESTION','GESTION','SUBGESTION','TIPO','RECORRIDO',
'CET','CONT_GEN','Hora_Llamada','Mejor_Telefono','TELEFONO','FECHA_LLAMADA','DIA',
'Ejecutivo','SUPERVISOR','segundos','VAL','AGENDADOS','SOLO FH1','SOLO FH2',
'SOLO FH3','DOBLE FH','TRIPLE FH','NUMXDNI','Estado','CntEstado','MntOferta',
'CNTVTAS','NUM_DIA_HABIL','Semana_Mes','tMontoDesem','CNT_LLAMADAS','AREAEFECTINEGOCIOS',    
'AÑO_DURACION_BASE','MES_DURACION_BASE','FECHA_ENVIO','SERVICIO','RETIRO',
'REP1','REP2','CONJUNTO3MESES','FLG_REP_MES_ANT','FLG_REP_MESES_ANT','FLG_SIN_ENR',
'tMesGestion','DESCRIPCION','DESCRIPCION2','llave','REGION','EDAD',
'RANGO_OFERTA','TIPO_DEUDA','MONTOREF','LOTE','CONJUNTO3MESES','SEGMICROPEQUENACOMERCIAL',
'TIPOBASEMICROPEQUENA','Proceso','Zona','DEPARTAMENTO', 
    ])


df_funel=set([
'NUMERO_DOCUMENTO','NRG','Dni','TIPO_GESTION','GESTION','SUBGESTION','TIPO','RECORRIDO','CET','CONT_GEN','Hora_Llamada','Mejor_Telefono','TELEFONO','FECHA_LLAMADA','DIA','Ejecutivo','RHFC','SUPERVISOR','segundos','VAL','AGENDADOS','SOLO FH1','SOLO FH2','SOLO FH3','DOBLE FH','TRIPLE FH','NUMXDNI','Estado','CntEstado','MntOferta','CNTVTAS','NUM_DIA_HABIL','Semana_Mes','Desembolso','CNT_LLAMADAS','AÑO_DURACION_BASE','MES_DURACION_BASE','FECHA_ENVIO','SERVICIO','RETIRO','tMesGestion','DESCRIPCION','DESCRIPCION2','llave','REGION','PERFIL','SEGMENTO','NOMCOMERCIAL','ZONA','DEPARTAMENTO','AGENCIA','RANGO_OFERTA','RANGO_HS_RS','RANGO_HS_PLUS','RANGO_TASA','SITUACIONLABORAL','TIPOINGRESO','FLGCONVENIO','FLGSUBPROCESO_HS','OFERTAREF','LOTE','FLG_REP_MES_ANT','FLG_REP_MESES_ANT','FLG_SIN_ENR','tMesGestion'   
    ])
base_real=set(['NUMDOCUMENTO','SEGMICROPEQUENACOMERCIAL','IMPDHM','TIPOBASEMICROPEQUENA','Nacimiento','Direccion','DEPARTAMENTO','PROVINCIA','DISTRITO','AREAEFECTINEGOCIOS','ruc','num_cuotas_pagadas','num_cuotas_impagas','prioridad','PERFILINICIAL','ULTIMOMONTODESEMBOLSADO','ULTIMOMONTODESEMBOLSADOPLUS20','empresa1','RangoSaldo1','empresa2','RangoSaldo2','empresa3','RangoSaldo3','flag_recurrencia_efectinegocio','CANAL_ASIGNADO','FECHA_ENVIO','MES_DURACION_BASE','AÑO_DURACION_BASE','SERVICIO','RETIRO','FLAT1','FLAT2','FLAT3','REP1','REP2','DEVUELTO','recompra','Proceso','Zona','Montos_Referenciales','REP3','REP4','MONTOREFERENCIALFT','MONTOREFERENCIALHS','flag_consentimiento','PERFIL_IC','DEUDAMYPESINEFE','DEUDAELECTRO','DEUDAMOTOS','DEUDAEFECTIVO'])

In [12]:
print(sorted(tb_general-df_funel ))
print(sorted(df_funel-tb_general ))


['EDAD', 'LINEA_Efectiva', 'MARCA', 'MARCA2', 'Nombre_Campana', 'Numero_Campana', 'PROVINCIA', 'Prioridad']
[]


In [9]:
# print([row['LINEA_FT' ] for row in df_base.select('LINEA_FT').distinct().collect()])
# print([row['LINEA_HS_RS' ] for row in df_base.select('LINEA_HS_RS').distinct().collect()])
print([row['TASA' ] for row in df_base.select('TASA').distinct().collect()])


['101', '69', '112', '73', '87', '113', '114.13', '85', '71', '98', '99', '110', '107', '96', '100', '70', '75', '78', '89', '77', '104', '90', '60', '68', '102', '95', '93', '103', '82', '92', '108', '86', '81', '97', '106', '67', '84', '79', '88', '105', '63', '65', '83', '109', '91', '94', '74', '72', '76', '80']


In [ ]:
['101', '69', '112', '73', '87', '113', '114.13', '85', '71', '98', '99', '110', '107', '96', '100', '70', '75', '78', '89', '77', '104', '90', '60', '68', '102', '95', '93', '103', '82', '92', '108', '86', '81', '97', '106', '67', '84', '79', '88', '105', '63', '65', '83', '109', '91', '94', '74', '72', '76', '80']


In [5]:
df_ges.show(3)

+----------------+---+-----------+--------+----------------+------------+-----------------+--------+-----------------+-----------------+------+----------+--------------+---------+---+--------+---------------+----------------+----------+----+------------+---------+-----------+-------+------+--------------+---------+--------+--------+--------+--------+---------+-------------+----+--------------+--------------------+------------------+---------+-------+-----------+---------------+-----------------+-----------+-----------+---------+--------+---+--------------+---------+------+---------+---------+-------+-------------+----------+-------------------+----------+------------+----+
|NUMERO_DOCUMENTO|NRG|FECHA_ENVIO|     Dni|    TIPO_GESTION|     GESTION|       SUBGESTION|SERVICIO|AÑO_DURACION_BASE|MES_DURACION_BASE|RETIRO|      TIPO|LINEA_Efectiva|RECORRIDO|CET|CONT_GEN|         REGION|    RANGO_OFERTA|RANGO_TASA|EDAD|Hora_Llamada|Prioridad|      MARCA| MARCA2|PERFIL|Mejor_Telefono| TELEFONO|SO

In [4]:
df_base.show(3)

+--------+--------------------+--------------------+-----+-----+------+------------+-----+--------------+---------------+--------+---------+------------+---------+-----------------+---------+------------+----+-------------+--------------+------+--------+-----+-------+--------------------+-------------------------+-------------------------+--------+--------+--------+--------+--------+--------+--------+--------+--------+---------+---------+---------+---------+---------+---------+--------------------+-----------+--------+-----------+--------+-----------+-----------+---------+---------+---------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-----------------+-----------------+--------+------+-----+------+------+------+------+-----+-----+-----+----+----+--------+----+----+----------+--------+----------------+-----------+--------+-----------+-------------+----------+----------------+------------------+-----

In [ ]:
[tGestionMesEfectiva]

In [25]:
print(sorted(tb_general-df_funel_debe_ir ))

['LINEA_EfectivaN', 'MARCA', 'MARCA2', 'Nombre_Campana', 'Numero_Campana', 'PERFIL', 'PROVINCIA', 'Prioridad', 'RANGO_TASA', 'REP3']


In [14]:
print(sorted(base_real - df_funel))


['AREAEFECTINEGOCIOS', 'AÑO_DURACION_BASE', 'CANAL_ASIGNADO', 'DEPARTAMENTO', 'DEUDAEFECTIVO', 'DEUDAELECTRO', 'DEUDAMOTOS', 'DEUDAMYPESINEFE', 'DEVUELTO', 'DISTRITO', 'Direccion', 'FECHA_ENVIO', 'FLAT1', 'FLAT2', 'FLAT3', 'IMPDHM', 'MES_DURACION_BASE', 'MONTOREFERENCIALFT', 'MONTOREFERENCIALHS', 'Montos_Referenciales', 'NUMDOCUMENTO', 'Nacimiento', 'PERFILINICIAL', 'PERFIL_IC', 'PROVINCIA', 'Proceso', 'REP1', 'REP2', 'REP3', 'REP4', 'RETIRO', 'RangoSaldo1', 'RangoSaldo2', 'RangoSaldo3', 'SEGMICROPEQUENACOMERCIAL', 'SERVICIO', 'TIPOBASEMICROPEQUENA', 'ULTIMOMONTODESEMBOLSADO', 'ULTIMOMONTODESEMBOLSADOPLUS20', 'Zona', 'empresa1', 'empresa2', 'empresa3', 'flag_consentimiento', 'flag_recurrencia_efectinegocio', 'num_cuotas_impagas', 'num_cuotas_pagadas', 'prioridad', 'recompra', 'ruc']


In [18]:
resultante=set(['NUMERO_DOCUMENTO','NRG','Dni','TIPO_GESTION','GESTION','SUBGESTION','TIPO','RECORRIDO','CET','CONT_GEN','Hora_Llamada','Mejor_Telefono','TELEFONO','FECHA_LLAMADA','DIA','Ejecutivo','SUPERVISOR','segundos','VAL','AGENDADOS','SOLO FH1','SOLO FH2','SOLO FH3','DOBLE FH','TRIPLE FH','NUMXDNI','Estado','CntEstado','MntOferta','CNTVTAS','NUM_DIA_HABIL','Semana_Mes','tMontoDesem','CNT_LLAMADAS','AREAEFECTINEGOCIOS','AÑO_DURACION_BASE','MES_DURACION_BASE','FECHA_ENVIO','SERVICIO','RETIRO','REP1','REP2','FLG_REP_MES_ANT','FLG_REP_MESES_ANT','FLG_SIN_ENR','tMesGestion','DESCRIPCION','DESCRIPCION2','llave','REGION','EDAD','RANGO_OFERTA','TIPO_DEUDA','MONTOREF','LOTE','CONJUNTO3MESES','SEGMICROPEQUENACOMERCIAL','TIPOBASEMICROPEQUENA','Proceso','Zona','DEPARTAMENTO'])

In [ ]:
['AREAEFECTINEGOCIOS', 'AÑO_DURACION_BASE', 'CANAL_ASIGNADO', 'DEPARTAMENTO', 'DEUDAEFECTIVO', 'DEUDAELECTRO', 'DEUDAMOTOS', 'DEUDAMYPESINEFE', 'DEVUELTO', 'DISTRITO', 'Direccion', 'FECHA_ENVIO', 'FLAT1', 'FLAT2', 'FLAT3', 'IMPDHM', 'MES_DURACION_BASE', 'MONTOREFERENCIALFT', 'MONTOREFERENCIALHS', 'Montos_Referenciales', 'NUMDOCUMENTO', 'Nacimiento', 'PERFILINICIAL', 'PERFIL_IC', 'PROVINCIA', 'Proceso', 'REP1', 'REP2', 'REP3', 'REP4', 'RETIRO', 'RangoSaldo1', 'RangoSaldo2', 'RangoSaldo3', 'SEGMICROPEQUENACOMERCIAL', 'SERVICIO', 'TIPOBASEMICROPEQUENA', 'ULTIMOMONTODESEMBOLSADO', 'ULTIMOMONTODESEMBOLSADOPLUS20', 'Zona', 'empresa1', 'empresa2', 'empresa3', 'flag_consentimiento', 'flag_recurrencia_efectinegocio', 'num_cuotas_impagas', 'num_cuotas_pagadas', 'prioridad', 'recompra', 'ruc']

In [5]:
columnas_extra = df_funel - tb_general
print(sorted(columnas_extra))

['RHFC']


In [4]:
columnas_comunes = tb_general & df_funel
print(columnas_comunes)

{'SOLO FH3', 'Dni', 'CET', 'CntEstado', 'NRG', 'SOLO FH1', 'Mejor_Telefono', 'CONT_GEN', 'CNTVTAS', 'NUMXDNI', 'NUM_DIA_HABIL', 'GESTION', 'TRIPLE FH', 'FECHA_LLAMADA', 'Hora_Llamada', 'segundos', 'DOBLE FH', 'TIPO', 'Semana_Mes', 'SUPERVISOR', 'Ejecutivo', 'RECORRIDO', 'SUBGESTION', 'AGENDADOS', 'tMontoDesem', 'TIPO_GESTION', 'DIA', 'VAL', 'NUMERO_DOCUMENTO', 'SOLO FH2', 'Estado', 'TELEFONO', 'MntOferta'}


In [2]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

In [10]:
query = f"""
select * from ODIN.dbo.Base_Maestra_Efectiva_Negocios_Vigente
"""
df_base=obtener_tabla_sql(spark,query,server_zeus,user_zeus,pwd_zeus,db_zeus)

In [15]:
print([row['PERFIL_IC' ] for row in df_base.select('PERFIL_IC').distinct().collect()])
print([row['PERFILINICIAL' ] for row in df_base.select('PERFILINICIAL').distinct().collect()])



[None]
[None]


In [11]:
columnas_faltantes =  tb_general- df_funel
print(sorted(columnas_faltantes))

['AREAEFECTINEGOCIOS', 'AÑO_DURACION_BASE', 'CNT_LLAMADAS', 'CONJUNTO3MESES', 'DESCRIPCION', 'DESCRIPCION2', 'EDAD', 'FECHA_ENVIO', 'FLG_REP_MESES_ANT', 'FLG_REP_MES_ANT', 'FLG_SIN_ENR', 'LINEA_EfectivaN', 'MARCA', 'MARCA2', 'MES_DURACION_BASE', 'Nombre_Campana', 'Numero_Campana', 'PERFIL', 'PROVINCIA', 'Prioridad', 'RANGO_OFERTA', 'RANGO_TASA', 'REGION', 'REP1', 'REP2', 'REP3', 'RETIRO', 'SERVICIO', 'llave', 'tMesGestion']


In [ ]:
xdxd

In [2]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from unidecode import unidecode
from sqlalchemy import create_engine
from sqlalchemy import text

fecha_mes_base='2026-07-01'
tipi_cond1='RECLUTAMIENTO'
servidor_01=64

campana='negocios'

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_zeus
db_sql = "ODIN"
user_sql = user_zeus
pwd_sql = pwd_zeus
engine_zeus = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_sa
db_sql = "ODIN"
user_sql = user_sa
pwd_sql = pwd_sa
engine_sa = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_zeus
db_sql = "SAMANTHA"
user_sql = user_zeus
pwd_sql = pwd_zeus
engine_samantha = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

def nombre_mes_anio(fecha_mes_base):
    from datetime import datetime

    fecha = datetime.strptime(fecha_mes_base, "%Y-%m-%d")

    meses = [
        "enero", "febrero", "marzo", "abril", "mayo", "junio",
        "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"
    ]

    return f"{meses[fecha.month - 1]} {fecha.year}"


In [5]:

ruta_archivo = os.path.join(ruta_ventas_desembolso, filename)
df_desembolso_cli = pd.read_excel(ruta_archivo,sheet_name='Base')

df_desembolso_cli["FECHA_DESEMBOLSO"] = pd.to_datetime(df_desembolso_cli["FECHA_DESEMBOLSO"], errors="coerce")

fecha_min = df_desembolso_cli["FECHA_DESEMBOLSO"].min()
print(fecha_min)
fecha_desembolso = fecha_min.replace(day=1)
fecha_desembolso = fecha_min.replace(day=1).strftime("%Y-%m-%d")
df_desembolso_cli['CAMPAÑA']=nombre_mes_anio(fecha_desembolso)


NameError: name 'filename' is not defined

In [8]:

from sqlalchemy import text

with engine_samantha.begin() as conn:
    conn.execute(text(f"""
        DELETE FROM SAMANTHA.dbo.efectiva_negocios_ventas_desembolso
        WHERE FECHA_DESEMBOLSO >= '{fecha_desembolso}'
          AND FECHA_DESEMBOLSO <= EOMONTH('{fecha_desembolso}');
    """))

    conn.execute(text(f"""
        DELETE FROM SAMANTHA.dbo.efectiva_negocios_ventas
        WHERE FECHA >= '{fecha_desembolso}'
          AND FECHA <= EOMONTH('{fecha_desembolso}');
    """))

df_desembolso_cli.to_sql(
    name="efectiva_negocios_ventas_desembolso",
    con=engine_samantha,
    if_exists="append",
    index=False,
    chunksize=1000
)




2

In [3]:

import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from unidecode import unidecode
from sqlalchemy import create_engine
from sqlalchemy import text

fecha_mes_base='2026-07-01'
tipi_cond1='RECLUTAMIENTO'
servidor_01=64

campana='negocios'

server_sql = server_kishin
db_sql = "DANTALION"
user_sql = user_kishin
pwd_sql = pwd_kishin

engine_kishin = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_zeus
db_sql = "ODIN"
user_sql = user_zeus
pwd_sql = pwd_zeus
engine_zeus = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_sa
db_sql = "ODIN"
user_sql = user_sa
pwd_sql = pwd_sa
engine_sa = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)
server_sql = server_zeus
db_sql = "SAMANTHA"
user_sql = user_zeus
pwd_sql = pwd_zeus
engine_samantha = create_engine(
    f"mssql+pyodbc://{user_sql}:{pwd_sql}@{server_sql}/{db_sql}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
)

def nombre_mes_anio(fecha_mes_base):
    from datetime import datetime

    fecha = datetime.strptime(fecha_mes_base, "%Y-%m-%d")

    meses = [
        "enero", "febrero", "marzo", "abril", "mayo", "junio",
        "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"
    ]

    return f"{meses[fecha.month - 1]} {fecha.year}"


In [ ]:
USE [MAEBA]
GO
/****** Object:  StoredProcedure [ADM_OBJ_TG].[spFunnelAlfin]    Script Date: 8/07/2026 00:47:01 ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO

ALTER PROCEDURE [ADM_OBJ_TG].[spFunnelAlfin]
as
	
DROP TABLE IF EXISTS
    #NumxDnixAlfin,
    #RhfAlfin,
    #FunnelAlfin,
	#Tmp_LLamadas_Alfin,
    #LlamadasAlfin1,
    #VentasAlfin,
    #Desembolso_alfin,
    #tempventa;

DECLARE @HOY DATE = GETDATE();
DECLARE @inicio DATE = DATEADD(DAY, 1, EOMONTH(@HOY, -1));


SELECT
    name AS Sinonimo,
    base_object_name AS Apunta_A
FROM sys.synonyms
WHERE name = 'synNumeroEfectivaNegocios';
/*====================================================================================================================================AR========================*/
/*============================================================================================================================================================*/
CREATE TABLE #RhfAlfin (RHF VARCHAR(3),[SOLO FH1]INT,[SOLO FH2]INT,[SOLO FH3]INT,[DOBLE FH]INT,[TRIPLE FH] INT)
INSERT INTO #RhfAlfin
VALUES  ('000',0,0,0,0,0),('100',1,0,0,0,0),('020',0,1,0,0,0),('003',0,0,1,0,0),('120',0,0,0,1,0),('103',0,0,0,1,0),('023',0,0,0,1,0),('123',0,0,0,0,1)

select NUMDOCUMENTO as DNI,count(*) as numxdni into #NumxDnixAlfin from [ADM_OBJ_TG].synNumeroAlfinBk group by NUMDOCUMENTO
create index NUMERO_DOCUMENTO on #NumxDnixAlfin(DNI)

SELECT DNI, COUNT(*) AS CNT_LLAMADAS 
INTO #LlamadasAlfin1 
FROM THOTH.dbo.Tmp_LLamadas_Alfin
--FROM THOTH.dbo.Tmp_LLamadas_Alfin_ref
GROUP BY DNI
create index DNI on #LlamadasAlfin1(DNI)

select
	Dni,
	Codigo_Paleta,
	Sub_Estado_,
	Estado_,
	Descripcion_,
	Hora_Llamada,
	Mejor_Telefono,
	PHONE_NUMBER as TELEFONO,
	FECHA_LLAMADA,
	segundos,
	RHFC,
	Ejecutivo,
	DNI_EJECUTIVO
into #Tmp_LLamadas_Alfin
from THOTH.dbo.Tmp_LLamadas_Alfin_5
--FROM THOTH.dbo.Tmp_LLamadas_Alfin_ref_5

SELECT
	TRY_CONVERT(
		INT,
		REPLACE(
			REPLACE(
				TRANSLATE(MONTO, '.,', '  '),
                    'S/ ',
                    ''
                ),
                ' ',
                ''
            )
        ) AS MONTO,
        DNI,
        EJECUTIVO,
        ESTADO,
        FECHA,
		row_number() OVER (PARTITION BY dni + ' ' + mes_Venta  
		ORDER BY fecha DESC, Trama_Hora DESC) ind_venta
    into #tempventa
	FROM SAMANTHA.dbo.Ventas_Target
	WHERE FECHA >= @inicio
	AND FECHA < DATEADD(DAY, 1, EOMONTH(@inicio))
	and CAMPANA='Negocios'
	AND DNI NOT IN ('') 
	AND LEN(LTRIM(RTRIM(DNI))) > 7


;WITH CTE AS
(
    SELECT
		MONTO,
        DNI,
        EJECUTIVO,
        ind_venta,
        ESTADO,
        FECHA,
        ROW_NUMBER() OVER (
            PARTITION BY DNI
            ORDER BY ind_venta DESC
        ) AS RN
    FROM #tempventa
	WHERE FECHA >= @inicio
	AND FECHA < DATEADD(DAY, 1, EOMONTH(@inicio))
)

SELECT
    MONTO,
    DNI,
    EJECUTIVO,
    ind_venta,
    ESTADO,
    FECHA
INTO #VentasAlfin
FROM CTE
WHERE RN = 1;

select 
DNI,
MONTO_NETO
into #Desembolso_alfin
from SAMANTHA.dbo.Alfin_ventas_desembolso
where FECHA_DESEMBOLSO>=@inicio
and FECHA_DESEMBOLSO<DATEADD(DAY, 1, EOMONTH(@inicio))

/*============================================================================================================================================================*/
/*============================================================================================================================================================*/
SELECT 
A.NUMDOCUMENTO NUMERO_DOCUMENTO,
1 AS NRG,
B.Dni,
case when isnull(B.Estado_,'') ='' then 'NO DISCADO' ELSE B.Estado_ END as TIPO_GESTION,
case when isnull(B.Sub_Estado_,'') ='' then 'NO DISCADO' ELSE B.Sub_Estado_ END as GESTION,
case when isnull(B.Descripcion_,'') ='' then 'NO DISCADO' ELSE B.Descripcion_ END as SUBGESTION,
CASE 
		WHEN ISNULL(a.FLAT2,0) =0 THEN 'NO ENRIQUECIDO'
		WHEN ISNULL(B.Codigo_Paleta,'')='' THEN  'NO DISCADO' 
		ELSE 'GEST+DISC' END as 'TIPO',
CASE  WHEN B.DNI IS NULL THEN 0 ELSE 1 END as RECORRIDO,
CASE WHEN B.Estado_ in ('CONTACTO EFECTIVO CON TITULAR','CONTACTO EFECTIVO') THEN 1 ELSE 0 END as CET,	
CASE WHEN B.Estado_ in ('CONTACTO EFECTIVO CON TITULAR',
						'CONTACTO EFECTIVO') 
					and not B.Descripcion_ in ('CLIENTE DESEA QUE NO VUELVAN A LLAMAR',
					'NO BRINDA CONSENTIMIENTO',
					'CLIENTE CORTO SIN ESCUCHAR OFERTA')THEN 1 ELSE 0 END as CONT_GEN,
b.Hora_Llamada,
b.Mejor_Telefono,
b.TELEFONO,
b.FECHA_LLAMADA,
substring(convert(varchar,b.FECHA_LLAMADA),9,2) as DIA,
isnull(d.tNombreCompleto,b.Ejecutivo) as Ejecutivo,
b.RHFC,
ISNULL(d.tSupervisor,'Sin_Asignar') as SUPERVISOR,
b.segundos, 
case 
when B.Estado_ in ('CONTACTO EFECTIVO CON TITULAR','CONTACTO EFECTIVO') and 
not B.Descripcion_ in ('CLIENTE DESEA QUE NO VUELVAN A LLAMAR','NO BRINDA CONSENTIMIENTO','CLIENTE CORTO SIN ESCUCHAR OFERTA') then 1 else 0 end AS VAL,
case when B.Descripcion_='VOLVER A LLAMAR' then 1 else 0 end as AGENDADOS,
cast(0 as int)[SOLO FH1],
cast(0 as int)[SOLO FH2],
cast(0 as int)[SOLO FH3],
cast(0 as int)[DOBLE FH],
cast(0 as int)[TRIPLE FH],
cast(0 as int)NUMXDNI,
cast(null as varchar(100)) Estado,
cast(null as NUMERIC) CntEstado,
cast(null as NUMERIC) MntOferta,
cast(null as int) CNTVTAS,
cast(null as int)[NUM_DIA_HABIL],
cast(null as varchar(100)) [Semana_Mes],
cast(null as float) tMontoDesem,
cast(null as float) CNT_LLAMADAS
into #FunnelAlfin
FROM [ADM_OBJ_TG].synBase_Maestra_Alfin_bk a 
LEFT JOIN #Tmp_LLamadas_Alfin b on a.NUMDOCUMENTO=b.Dni
LEFT JOIN [ADM_OBJ_TG].[synUsuariosTg] d ON b.DNI_EJECUTIVO=d.tDocumentoVici COLLATE Modern_Spanish_CI_AS

/*============================================================================================================================================================*/
update a 
set a.[SOLO FH1]=ISNULL(b.[SOLO FH1],0),
	a.[SOLO FH2]=ISNULL(b.[SOLO FH2],0),
	a.[SOLO FH3]=ISNULL(b.[SOLO FH3],0),
	a.[DOBLE FH]=ISNULL(b.[DOBLE FH],0),
	a.[TRIPLE FH] =ISNULL(b.[TRIPLE FH],0)
from #FunnelAlfin a 
inner join #RhfAlfin b ON a.RHFC=b.RHF COLLATE Modern_Spanish_CI_AS 
where a.NUMERO_DOCUMENTO IS NOT NULL

update a 
set a.NUMXDNI=b.NUMXDNI
from #FunnelAlfin a 
LEFT JOIN #NumxDnixAlfin b ON a.NUMERO_DOCUMENTO=b.DNI
where a.NUMERO_DOCUMENTO IS NOT NULL

update a 
set a.TIPO_GESTION  ='CONTACTO EFECTIVO', 
	a.GESTION ='EFECTIVO', 
	a.SUBGESTION ='VOLVER A LLAMAR'
from #FunnelAlfin a left join #VentasAlfin b 
on a.NUMERO_DOCUMENTO=b.dni COLLATE Modern_Spanish_CI_AS
where b.dni is null and a.SUBGESTION='SI QUIERE'
UPDATE a
SET a.TIPO_GESTION='CONTACTO EFECTIVO',
	a.GESTION ='EFECTIVO',
	a.SUBGESTION='SI QUIERE'
from #FunnelAlfin a 
inner join #VentasAlfin b on a.NUMERO_DOCUMENTO=b.dni COLLATE Modern_Spanish_CI_AS
UPDATE a SET a.Ejecutivo=isnull(c.tNombreCompleto,b.EJECUTIVO)
FROM #FunnelAlfin a 
inner join #VentasAlfin b on a.NUMERO_DOCUMENTO=b.dni COLLATE Modern_Spanish_CI_AS
inner join [ADM_OBJ_TG].[synUsuariosTg] c on b.dni=c.tDocumentoVici COLLATE Modern_Spanish_CI_AS
UPDATE  a SET a.MntOferta =b.MONTO
FROM #FunnelAlfin a inner join #VentasAlfin b 
on a.NUMERO_DOCUMENTO=b.dni COLLATE Modern_Spanish_CI_AS;
UPDATE  a SET a.CNTVTAS =1 --, MntOferta=b.monto
FROM #FunnelAlfin a inner join #VentasAlfin b 
on a.NUMERO_DOCUMENTO=b.dni COLLATE Modern_Spanish_CI_AS;
UPDATE a SET a.Estado=isnull(replace(b.ESTADO,'-','EN PROCESO'),'EN PROCESO')
FROM #FunnelAlfin a inner join #VentasAlfin b 
on a.NUMERO_DOCUMENTO=b.dni COLLATE Modern_Spanish_CI_AS;
UPDATE a SET a.CntEstado=1
FROM #FunnelAlfin a inner join #VentasAlfin b 
on a.NUMERO_DOCUMENTO=b.dni COLLATE Modern_Spanish_CI_AS 
where b.ESTADO='1.-VALIDADA';
UPDATE a SET a.FECHA_LLAMADA =convert(varchar,CONVERT(date,b.FECHA)), a.DIA=substring(convert(varchar,b.FECHA),9,2)
FROM #FunnelAlfin a, #VentasAlfin b 
WHERE A.NUMERO_DOCUMENTO =B.dni COLLATE Modern_Spanish_CI_AS;

update a set a.[NUM_DIA_HABIL]=b.[NUM_DIA_HABIL], a.Semana_Mes=b.Semana_Mes from #FunnelAlfin a inner join [ADM_OBJ_TG].[synDiaHabil] b on a.fecha_llamada=b.Fecha

update a 
set a.tMontoDesem = b.MONTO_NETO
from #FunnelAlfin a inner join #Desembolso_alfin b 
on a.NUMERO_DOCUMENTO=b.DNI COLLATE Modern_Spanish_CI_AS

update a 
set a.CNT_LLAMADAS= b.CNT_LLAMADAS
from #FunnelAlfin a inner join #LlamadasAlfin1 b 
on a.NUMERO_DOCUMENTO=b.DNI COLLATE Modern_Spanish_CI_AS

/*============================================================================================================================================================*/
/*============================================================================================================================================================*/

delete from [ADM_OBJ_TG].[tGestionMesEfectivaN] where tMesGestion=DATENAME(MONTH,@inicio) and AÑO_DURACION_BASE=YEAR(@inicio);
insert into [ADM_OBJ_TG].[tGestionMesEfectivaN](
NUMERO_DOCUMENTO,NRG,Dni,TIPO_GESTION,GESTION,SUBGESTION,TIPO,RECORRIDO,
CET,CONT_GEN,Hora_Llamada,Mejor_Telefono,TELEFONO,FECHA_LLAMADA,DIA,
Ejecutivo,RHFC,SUPERVISOR,segundos,VAL,AGENDADOS,[SOLO FH1],[SOLO FH2],
[SOLO FH3],[DOBLE FH],[TRIPLE FH],NUMXDNI,Estado,CntEstado,MntOferta,
CNTVTAS,NUM_DIA_HABIL,Semana_Mes,tMontoDesem,CNT_LLAMADAS
,AREAEFECTINEGOCIOS,AÑO_DURACION_BASE,MES_DURACION_BASE,
FECHA_ENVIO,SERVICIO,RETIRO,
REP1,REP2,CONJUNTO3MESES,FLG_REP_MES_ANT,FLG_REP_MESES_ANT,FLG_SIN_ENR,
tMesGestion,DESCRIPCION,DESCRIPCION2,llave,REGION,EDAD,
RANGO_OFERTA,TIPO_DEUDA,MONTOREF,LOTE,SEGMICROPEQUENACOMERCIAL,
TIPOBASEMICROPEQUENA,Proceso,Zona,DEPARTAMENTO)

select a.*,b.AREAEFECTINEGOCIOS,b.AÑO_DURACION_BASE,b.MES_DURACION_BASE,
b.FECHA_ENVIO,b.SERVICIO,b.RETIRO,
CASE WHEN b.REP1=1 THEN 'Stock' ELSE 'NUEVO' END AS REP1,
CASE WHEN b.REP2=1 THEN 'Stock' ELSE 'NUEVO' END AS REP2,
CASE WHEN (b.REP1=1 OR b.REP2=1) 
THEN 'Stock' ELSE 'Nuevo' END AS CONJUNTO3MESES,
FLG_REP_MES_ANT=CASE WHEN ISNULL(REP1,0)=1 THEN 'Stock' Else 'Nuevo' END,
FLG_REP_MESES_ANT=CASE WHEN ISNULL(REP2,0)=1 THEN 'Stock' Else 'Nuevo' END, 
FLG_SIN_ENR =CASE WHEN ISNULL(FLAT2,0) =0 THEN 'SIN ENRIQUECER' END,
DATENAME(MONTH,@inicio) AS tMesGestion,
'REGISTROS ENTREGADOS'  AS 'DESCRIPCION',
'REGISTROS CARGADOS'  AS 'DESCRIPCION2',
a.NUMERO_DOCUMENTO+' '+convert(varchar,fecha_envio) as llave,
REGION=CASE	WHEN ISNULL(b.[PROVINCIA],'') ='' THEN 'Leads sin región'
			WHEN b.[PROVINCIA] ='LIMA' OR b.[PROVINCIA] ='CALLAO' THEN 'Leads Lima'
			WHEN b.[PROVINCIA] <>'LIMA' THEN 'Leads Provincia' end,

EDAD =
    CASE
        WHEN b.Nacimiento IS NULL THEN 'SIN DATO'
        WHEN DATEDIFF(YEAR, b.Nacimiento, GETDATE()) BETWEEN 18 AND 29 THEN '18-29'
        WHEN DATEDIFF(YEAR, b.Nacimiento, GETDATE()) BETWEEN 30 AND 39 THEN '30-39'
        WHEN DATEDIFF(YEAR, b.Nacimiento, GETDATE()) BETWEEN 40 AND 49 THEN '40-49'
        WHEN DATEDIFF(YEAR, b.Nacimiento, GETDATE()) BETWEEN 50 AND 59 THEN '50-59'
        WHEN DATEDIFF(YEAR, b.Nacimiento, GETDATE()) >= 60 THEN '60 A MAS'
        ELSE 'MENOR 18'
    END,
RANGO_OFERTA =
CASE
    WHEN TRY_CONVERT(NUMERIC(18,2), REPLACE(b.IMPDHM, ',', '')) IS NULL THEN 'SIN DATO'
    WHEN TRY_CONVERT(NUMERIC(18,2), REPLACE(b.IMPDHM, ',', '')) < 10000 THEN '00.<0 - 10,000]'
    WHEN TRY_CONVERT(NUMERIC(18,2), REPLACE(b.IMPDHM, ',', '')) < 20000 THEN '01.<10,000 - 20,000]'
    WHEN TRY_CONVERT(NUMERIC(18,2), REPLACE(b.IMPDHM, ',', '')) < 30000 THEN '02.<20,000 - 30,000]'
    WHEN TRY_CONVERT(NUMERIC(18,2), REPLACE(b.IMPDHM, ',', '')) < 40000 THEN '03.<30,000 - 40,000]'
    WHEN TRY_CONVERT(NUMERIC(18,2), REPLACE(b.IMPDHM, ',', '')) < 50000 THEN '04.<40,000 - 50,000]'
    WHEN TRY_CONVERT(NUMERIC(18,2), REPLACE(b.IMPDHM, ',', '')) < 60000 THEN '05.<50,000 - 60,000]'
    WHEN TRY_CONVERT(NUMERIC(18,2), REPLACE(b.IMPDHM, ',', '')) < 70000 THEN '06.<60,000 - 70,000]'
    WHEN TRY_CONVERT(NUMERIC(18,2), REPLACE(b.IMPDHM, ',', '')) < 80000 THEN '07.<70,000 - 80,000]'
    WHEN TRY_CONVERT(NUMERIC(18,2), REPLACE(b.IMPDHM, ',', '')) < 90000 THEN '08.<80,000 - 90,000]'
    WHEN TRY_CONVERT(NUMERIC(18,2), REPLACE(b.IMPDHM, ',', '')) < 100000 THEN '09.<90,000 - 100,000]'
    WHEN TRY_CONVERT(NUMERIC(18,2), REPLACE(b.IMPDHM, ',', '')) < 150000 THEN '10.<100,000 - 150,000]'
    WHEN TRY_CONVERT(NUMERIC(18,2), REPLACE(b.IMPDHM, ',', '')) < 200000 THEN '11.<150,000 - 200,000]'
    WHEN TRY_CONVERT(NUMERIC(18,2), REPLACE(b.IMPDHM, ',', '')) <= 250000 THEN '12.<200,000 - 250,000]'
    ELSE '13.<250,000 A MÁS]'
END,
TIPO_DEUDA=CASE
    WHEN DEUDAMYPESINEFE IS NULL
     AND DEUDAELECTRO IS NULL
     AND DEUDAMOTOS IS NULL
     AND DEUDAEFECTIVO IS NULL
        THEN 'SIN DATO'
    WHEN ISNULL(DEUDAMYPESINEFE,0) >= ISNULL(DEUDAELECTRO,0)
     AND ISNULL(DEUDAMYPESINEFE,0) >= ISNULL(DEUDAMOTOS,0)
     AND ISNULL(DEUDAMYPESINEFE,0) >= ISNULL(DEUDAEFECTIVO,0)
        THEN 'DEUDAMYPESINEFE'
    WHEN ISNULL(DEUDAELECTRO,0) >= ISNULL(DEUDAMYPESINEFE,0)
     AND ISNULL(DEUDAELECTRO,0) >= ISNULL(DEUDAMOTOS,0)
     AND ISNULL(DEUDAELECTRO,0) >= ISNULL(DEUDAEFECTIVO,0)
        THEN 'DEUDAELECTRO'
    WHEN ISNULL(DEUDAMOTOS,0) >= ISNULL(DEUDAMYPESINEFE,0)
     AND ISNULL(DEUDAMOTOS,0) >= ISNULL(DEUDAELECTRO,0)
     AND ISNULL(DEUDAMOTOS,0) >= ISNULL(DEUDAEFECTIVO,0)
        THEN 'DEUDAMOTOS'
    ELSE 'DEUDAEFECTIVO'
END,
MONTOREF=CASE
    WHEN MONTOREFERENCIALFT IS NULL
     AND MONTOREFERENCIALHS IS NULL
        THEN 'SIN DATO'

    WHEN MONTOREFERENCIALFT IS NOT NULL
        THEN 'FT'

    WHEN MONTOREFERENCIALHS IS NOT NULL
        THEN 'HS'
END,
LOTE=CASE	WHEN b.FECHA_ENVIO ='2026-07-01' THEN 'BASE REGULAR'
			WHEN b.FECHA_ENVIO ='2026-06-01' THEN 'BASE REGULAR'
			WHEN b.FECHA_ENVIO ='2026-05-01' THEN 'BASE REGULAR'
			WHEN b.FECHA_ENVIO ='2026-05-02' THEN 'BASE PILOTO'
			WHEN b.FECHA_ENVIO ='2026-07-02' THEN 'BASE PILOTO'
			WHEN b.FECHA_ENVIO ='2026-06-02' THEN 'BASE PILOTO'
			end,
b.SEGMICROPEQUENACOMERCIAL,b.TIPOBASEMICROPEQUENA,b.Proceso,b.Zona,
b.DEPARTAMENTO
from #FunnelAlfin a
inner join [ADM_OBJ_TG].synBase_Maestra_Alfin_bk b
on a.NUMERO_DOCUMENTO=b.NUMDOCUMENTO

--odin.dbo.Sp_Actualizar_Alfin

/*============================================================================================================================================================*/
/*============================================================================================================================================================================================*/
/*============================================================================================================================================================================================*/
/*============================================================================================================================================================================================*/
/*============================================================================================================================================================================================*/
/*============================================================================================================================================================================================*/
/*
If object_id('[ADM_OBJ_TG].VentasEfectivaNFunnel') is not null
Drop table [ADM_OBJ_TG].VentasEfectivaNFunnel;

DECLARE @MES1 VARCHAR(100)
DECLARE @MES2 VARCHAR(100)
DECLARE @MES3 VARCHAR(100)

SET @MES1 = DATENAME(MONTH, DATEADD(MM,0,GETDATE()))
SET @MES2 = DATENAME(MONTH, DATEADD(MM,-1,GETDATE()))
SET @MES3 = DATENAME(MONTH, DATEADD(MM,-2,GETDATE()))

--select @MES1,@MES2,@MES3
--select * from [ADM_OBJ_TG].[synVentasEfectivaNFunnel]
--select DATENAME(MONTH, DATEADD(MM,0,GETDATE()))

select vnt.*,cast(dt.NUM_DIA_HABIL as int) dia_,dt.[Semana_Mes],tp.[tSupervisor]
into [ADM_OBJ_TG].VentasEfectivaNFunnel
from [ADM_OBJ_TG].[synVentasEfectivaNFunnel] VNT 
INNER JOIN [ADM_OBJ_TG].[synDiaHabil] DT ON VNT.FECHA=DT.Fecha
inner join [ADM_OBJ_TG].[synUsuariosTg] tp on vnt.DNIEjecutivo=tp.tDocumentoVici
WHERE --VNT.FECHA between '20231101' and '20240331' 
DATENAME(MONTH,VNT.FECHA) in (@MES1,@MES2,@MES3)
and CAMPANA='Negocios'
order by FECHA

--where MES_VENTA in (@MES1,@MES2,@MES3) and CAMPANA='Negocios'
--AND ANIO=2024
--order by FECHA

--SELECT * FROM [ADM_OBJ_TG].VentasEfectivaNFunnel
-- select * from [ADM_OBJ_TG].[synVentasEfectivaNFunnel]
/*============================================================================================================================================================*/
If object_id('ADM_OBJ_TG.PassingEfectivaNFunnel') is not null
Drop table ADM_OBJ_TG.PassingEfectivaNFunnel;

SELECT fecha,MES_VENTA,dia,EJECUTIVO,ESTADO,count(*) registros 
into ADM_OBJ_TG.PassingEfectivaNFunnel
FROM [ADM_OBJ_TG].[synVentasEfectivaNFunnel]
group by fecha,MES_VENTA,dia,EJECUTIVO,ESTADO

/*============================================================================================================================================================*/
SELECT A.*,B.DIAS_UTILES 
INTO ADM_OBJ_TG.FunnelDiasUtilesEfectivaN
FROM
(SELECT DATENAME(MONTH, DATEADD(MM,0,GETDATE())) MES,MAX(NUM_DIA_HABIL) DIA_HABIL 
FROM [MAEBA].[ADM_OBJ_TG].[tGestionMesEfectivaN] WHERE tMesGestion=DATENAME(MONTH, DATEADD(MM,0,GETDATE()))
AND FECHA_LLAMADA>=DATEADD(DAY, 1, EOMONTH(GETDATE(), - 1))) A
INNER JOIN
(SELECT DATENAME(MONTH, DATEADD(MM,0,GETDATE())) MES,COUNT(*) DIAS_UTILES 
FROM [ADM_OBJ_TG].[synDiaHabil] WHERE Fecha_Txt LIKE (LEFT(DATEADD(DAY, 1, EOMONTH(GETDATE(), - 1)),7)+'%') AND Nombre_dia!='Domingo') B ON A.MES=B.MES

SELECT tMesGestion,SUM(NRG) LEADS,sum(tMontoDesem) VENTAS,SUM(RECORRIDO) RECORRIDOS,SUM(CET) CET,cast(SUM(CNT_LLAMADAS) as decimal(20,4)) CNT_LLAMADAS
INTO ADM_OBJ_TG.FunnelResumenEfectivaN
FROM [MAEBA].[ADM_OBJ_TG].[tGestionMesEfectivaN] AVC 
WHERE tMesGestion=DATENAME(MONTH, DATEADD(MM,0,GETDATE())) and AÑO_DURACION_BASE=YEAR(getdate())
GROUP BY tMesGestion

If object_id('ADM_OBJ_TG.FunnelResumenEFEN') is not null
Drop table ADM_OBJ_TG.FunnelResumenEFEN;
SELECT A.*,D.DIA_HABIL,D.DIAS_UTILES,
CAST((VENTAS/DIA_HABIL) AS INT)*DIAS_UTILES AS PROYECCION,
cast(CNT_LLAMADAS/RECORRIDOS as decimal(10,4)) Intensidad
INTO ADM_OBJ_TG.FunnelResumenEFEN
FROM ADM_OBJ_TG.FunnelResumenEfectivaN A LEFT JOIN ADM_OBJ_TG.FunnelDiasUtilesEfectivaN D
ON A.tMesGestion=D.MES

/*============================================================================================================================================================*/
If object_id('adm_obj_tg.RhfEfectivaN') is not null
Drop table adm_obj_tg.RhfEfectivaN;
If object_id('adm_obj_tg.NumxDnixEfectivaN') is not null
Drop table adm_obj_tg.NumxDnixEfectivaN;
If object_id('adm_obj_tg.FunnelEfectivaN') is not null
Drop table adm_obj_tg.FunnelEfectivaN;
If object_id('adm_obj_tg.LlamadasEfectivaN1') is not null
Drop table adm_obj_tg.LlamadasEfectivaN1;
If object_id('adm_obj_tg.LlamadasEfectivaN2') is not null
Drop table adm_obj_tg.LlamadasEfectivaN2;
If object_id('ADM_OBJ_TG.FunnelResumenEfectivaN') is not null
Drop table ADM_OBJ_TG.FunnelResumenEfectivaN;
If object_id('ADM_OBJ_TG.FunnelDiasUtilesEfectivaN') is not null
Drop table ADM_OBJ_TG.FunnelDiasUtilesEfectivaN;
If object_id('tDesembolso_efeneg') is not null
Drop table tDesembolso_efeneg;


*/

In [4]:
filename='TARGET_EFECTIVO_202607.xlsx'

ruta_archivo = os.path.join(ruta_ventas_desembolso, filename)
df_desembolso_cli = pd.read_excel(ruta_archivo,sheet_name='Base')

df_desembolso_cli["FECHA_HORA_DESEMBOLSO"] = pd.to_datetime(df_desembolso_cli["FECHA_HORA_DESEMBOLSO"], errors="coerce")

fecha_min = df_desembolso_cli["FECHA_HORA_DESEMBOLSO"].min()
print(fecha_min)
fecha_desembolso = fecha_min.replace(day=1)
fecha_desembolso = fecha_min.replace(day=1).strftime("%Y-%m-%d")
df_desembolso_cli['CAMPAÑA']=nombre_mes_anio(fecha_desembolso)


from sqlalchemy import text

with engine_samantha.begin() as conn:
    conn.execute(text(f"""
        DELETE FROM SAMANTHA.dbo.efectiva_ventas_desembolso
        WHERE FECHA_HORA_DESEMBOLSO >= '{fecha_desembolso}'
          AND FECHA_HORA_DESEMBOLSO <= EOMONTH('{fecha_desembolso}');
    """))

df_desembolso_cli.to_sql(
    name="efectiva_ventas_desembolso",
    con=engine_samantha,
    if_exists="append",
    index=False,
    chunksize=1000
)





2026-07-01 00:00:00


13

In [5]:
filename='Target_Efectinegocio_202607.xlsx'

ruta_archivo = os.path.join(ruta_ventas_desembolso, filename)
df_desembolso_cli = pd.read_excel(ruta_archivo,sheet_name='Base')

df_desembolso_cli["FECHA_DESEMBOLSO"] = pd.to_datetime(df_desembolso_cli["FECHA_DESEMBOLSO"], errors="coerce")

fecha_min = df_desembolso_cli["FECHA_DESEMBOLSO"].min()
print(fecha_min)
fecha_desembolso = fecha_min.replace(day=1)
fecha_desembolso = fecha_min.replace(day=1).strftime("%Y-%m-%d")
df_desembolso_cli['CAMPAÑA']=nombre_mes_anio(fecha_desembolso)


from sqlalchemy import text

with engine_samantha.begin() as conn:
    conn.execute(text(f"""
        DELETE FROM SAMANTHA.dbo.efectiva_negocios_ventas_desembolso
        WHERE FECHA_DESEMBOLSO >= '{fecha_desembolso}'
          AND FECHA_DESEMBOLSO <= EOMONTH('{fecha_desembolso}');
    """))

df_desembolso_cli.to_sql(
    name="efectiva_negocios_ventas_desembolso",
    con=engine_samantha,
    if_exists="append",
    index=False,
    chunksize=1000
)





2026-07-02 00:00:00


10

In [ ]:
xdxdxd

In [ ]:
USE [MAEBA]
GO
/****** Object:  StoredProcedure [ADM_OBJ_TG].[spFunnelDinersTc]    Script Date: 5/07/2026 16:36:06 ******/
SET ANSI_NULLS ON
GO
SET QUOTED_IDENTIFIER ON
GO


ALTER PROCEDURE [ADM_OBJ_TG].[spFunnelDinersTc]
as

DROP TABLE IF EXISTS
    #NumxDnixDinersTc,
    #RhfRhfDinersTc,
    #FunnelDinersTc,
	#Tmp_LLamadas_Diners_TC,
    #LlamadasDinersTc1,
    #VentasDinersTc,
    #tempventa,
	dbo.LlamadasEfectivaN2;

DECLARE @HOY DATE = GETDATE();
DECLARE @inicio DATE = DATEADD(DAY, 1, EOMONTH(@HOY, -1));


/*====================================================================================================================================AR========================*/
/*============================================================================================================================================================*/
CREATE TABLE #RhfRhfDinersTc (RHF VARCHAR(3),[SOLO FH1]INT,[SOLO FH2]INT,[SOLO FH3]INT,[DOBLE FH]INT,[TRIPLE FH] INT)
INSERT INTO #RhfRhfDinersTc
VALUES  ('000',0,0,0,0,0),('100',1,0,0,0,0),('020',0,1,0,0,0),('003',0,0,1,0,0),('120',0,0,0,1,0),('103',0,0,0,1,0),('023',0,0,0,1,0),('123',0,0,0,0,1)

select NUMERO_DOCUMENTO as DNI,count(*) as numxdni into #NumxDnixDinersTc from [ADM_OBJ_TG].[synNumeroDinersTc] group by NUMERO_DOCUMENTO
create index NUMERO_DOCUMENTO on #NumxDnixDinersTc(DNI)

SELECT DNI, COUNT(*) AS CNT_LLAMADAS 
INTO #LlamadasDinersTc1 
FROM THOTH.dbo.Tmp_LLamadas_Diners_TC
--FROM THOTH.dbo.Tmp_LLamadas_Diners_TC_ref
GROUP BY DNI
create index DNI on #LlamadasDinersTc1(DNI)

select
	Dni,
	Codigo_Paleta,
	Sub_Estado_,
	Estado_,
	Descripcion_,
	Hora_Llamada,
	Mejor_Telefono,
	PHONE_NUMBER as TELEFONO,
	FECHA_LLAMADA,
	segundos,
	RHFC,
	Ejecutivo,
	DNI_EJECUTIVO
into #Tmp_LLamadas_Diners_TC
from THOTH.dbo.Tmp_LLamadas_Diners_TC_5
--FROM THOTH.dbo.Tmp_LLamadas_Diners_TC_ref_5

SELECT
	TRY_CONVERT(
		INT,
		REPLACE(
			REPLACE(
				TRANSLATE(MONTO, '.,', '  '),
                    'S/ ',
                    ''
                ),
                ' ',
                ''
            )
        ) AS MONTO,
        DNI,
        EJECUTIVO,
        ESTADO,
        FECHA,
		row_number() OVER (PARTITION BY dni + ' ' + mes_Venta  
		ORDER BY fecha DESC, Trama_Hora DESC) ind_venta
    into #tempventa
	FROM SAMANTHA.dbo.Ventas_Target
	WHERE FECHA >= @inicio
	AND FECHA < DATEADD(DAY, 1, EOMONTH(@inicio))
	and CAMPANA='DinersTc'
	AND DNI NOT IN ('') 
	AND LEN(LTRIM(RTRIM(DNI))) > 7

;WITH CTE AS
(
    SELECT
		MONTO,
        DNI,
        EJECUTIVO,
        ind_venta,
        ESTADO,
        FECHA,
        ROW_NUMBER() OVER (
            PARTITION BY DNI
            ORDER BY ind_venta DESC
        ) AS RN
    FROM #tempventa
	WHERE FECHA >= @inicio
	AND FECHA < DATEADD(DAY, 1, EOMONTH(@inicio))
)

SELECT
    MONTO,
    DNI,
    EJECUTIVO,
    ind_venta,
    ESTADO,
    FECHA
INTO #VentasDinersTc
FROM CTE
WHERE RN = 1;


/*============================================================================================================================================================*/
/*============================================================================================================================================================*/
SELECT 
A.NUMERO_DOCUMENTO,
1 AS NRG,
B.Dni,
TIPO_GESTION= case when isnull(B.Estado_,'') ='' then 'NO DISCADO' ELSE B.Estado_ END,
GESTION= case when isnull(B.Sub_Estado_,'') ='' then 'NO DISCADO' ELSE B.Sub_Estado_ END,
SUBGESTION= case when isnull(B.Descripcion_,'') ='' then 'NO DISCADO' ELSE B.Descripcion_ END,
'TIPO'=CASE 
		WHEN ISNULL(FLAT2,0) =0 THEN 'NO PHONENUMBER'
		WHEN ISNULL(B.Codigo_Paleta,'')='' THEN  'NO DISCADO' 
		ELSE 'GEST+DISC' END,
RECORRIDO=CASE  WHEN B.DNI IS NULL THEN 0 ELSE 1 END,
CET=CASE WHEN B.Estado_ in ('CONTACTO EFECTIVO CON TITULAR','CONTACTO EFECTIVO') THEN 1 ELSE 0 END,	
CONT_GEN=
case 
	when 
	  b.[Sub_Estado_] in ('NO ESCUCHO OFERTA',
	  'CONTACTO CON TERCERO','FALLECIÓ','ABANDONO','LINEA INOPERATIVA',
	  'LINEA OPERATIVA') and b.DNI_Ejecutivo <> 'VDAD' then 1 
	  else 0 end,
b.Hora_Llamada,
b.Mejor_Telefono,
b.TELEFONO,
b.Fecha_Llamada AS FECHA_LLAMADA,
substring(convert(varchar,b.Fecha_Llamada),9,2) as DIA,
isnull(d.tNombreCompleto,b.Ejecutivo) as Ejecutivo,
ISNULL(d.tSupervisor,'Sin_Asignar')  as SUPERVISOR,
b.segundos , 
case 
when B.Estado_ in ('CONTACTO EFECTIVO CON TITULAR','CONTACTO EFECTIVO') and 
b.Sub_Estado_ not like '%NO ESCUCHO OFERTA%'  then 1 else 0 end AS VAL,
case when B.Descripcion_='CITA TELEFONICA CON TITULAR' then 1 else 0 end as AGENDADOS,
b.DNI_EJECUTIVO,
b.RHFC,
cast(0 as int)[SOLO FH1],
cast(0 as int)[SOLO FH2],
cast(0 as int)[SOLO FH3],
cast(0 as int)[DOBLE FH],
cast(0 as int)[TRIPLE FH],
cast(0 as int)NUMXDNI,
cast(null as varchar(100)) as Estado,
cast(null as NUMERIC) as CntEstado,
cast(null as NUMERIC) as MntOferta,
cast(null as int) as CNTVTAS,
cast(null as int)as [NUM_DIA_HABIL],
cast(null as varchar(100)) as [Semana_Mes],
cast(null as float) as CNT_LLAMADAS
into #FunnelDinersTc
FROM [ADM_OBJ_TG].[synBaseMaestraDinersTc] a 
LEFT JOIN #Tmp_LLamadas_Diners_TC b on a.NUMERO_DOCUMENTO=b.Dni
LEFT JOIN [ADM_OBJ_TG].[synUsuariosTg] d ON b.DNI_EJECUTIVO=d.tDocumentoVici COLLATE Modern_Spanish_CI_AS

/*============================================================================================================================================================*/
update a 
set a.[SOLO FH1]=ISNULL(b.[SOLO FH1],0),
	a.[SOLO FH2]=ISNULL(b.[SOLO FH2],0),
	a.[SOLO FH3]=ISNULL(b.[SOLO FH3],0),
	a.[DOBLE FH]=ISNULL(b.[DOBLE FH],0),
	a.[TRIPLE FH] =ISNULL(b.[TRIPLE FH],0)
from #FunnelDinersTc a 
inner join #RhfRhfDinersTc b ON a.RHFC=b.RHF COLLATE Modern_Spanish_CI_AS 
where a.NUMERO_DOCUMENTO IS NOT NULL

update a 
set a.NUMXDNI=b.NUMXDNI
from #FunnelDinersTc a 
LEFT JOIN #NumxDnixDinersTc b ON a.NUMERO_DOCUMENTO=b.DNI
where a.NUMERO_DOCUMENTO IS NOT NULL

update a 
set a.TIPO_GESTION  ='CONTACTO EFECTIVO', 
	a.GESTION ='EFECTIVO', 
	a.SUBGESTION ='VOLVER A LLAMAR'
from #FunnelDinersTc a left join #VentasDinersTc b 
on a.NUMERO_DOCUMENTO=b.dni COLLATE Modern_Spanish_CI_AS
where b.dni is null and a.SUBGESTION='SI QUIERE'

UPDATE a
SET a.TIPO_GESTION='CONTACTO EFECTIVO',
	a.GESTION ='EFECTIVO',
	a.SUBGESTION='SI QUIERE'
from #FunnelDinersTc a 
inner join #VentasDinersTc b on a.NUMERO_DOCUMENTO=b.dni COLLATE Modern_Spanish_CI_AS

UPDATE a SET a.Ejecutivo=isnull(c.tNombreCompleto,b.EJECUTIVO)
FROM #FunnelDinersTc a 
inner join #VentasDinersTc b on a.NUMERO_DOCUMENTO=b.dni COLLATE Modern_Spanish_CI_AS
inner join [ADM_OBJ_TG].[synUsuariosTg] c on b.dni=c.tDocumentoVici COLLATE Modern_Spanish_CI_AS

UPDATE  a SET a.MntOferta =b.MONTO
FROM #FunnelDinersTc a inner join #VentasDinersTc b 
on a.NUMERO_DOCUMENTO=b.dni COLLATE Modern_Spanish_CI_AS;

UPDATE  a SET a.CNTVTAS =1 --, MntOferta=b.monto
FROM #FunnelDinersTc a inner join #VentasDinersTc b 
on a.NUMERO_DOCUMENTO=b.dni COLLATE Modern_Spanish_CI_AS;

UPDATE a SET a.Estado=isnull(replace(b.ESTADO,'-','EN PROCESO'),'EN PROCESO')
FROM #FunnelDinersTc a inner join #VentasDinersTc b 
on a.NUMERO_DOCUMENTO=b.dni COLLATE Modern_Spanish_CI_AS;

UPDATE a SET a.CntEstado=1
FROM #FunnelDinersTc a inner join #VentasDinersTc b 
on a.NUMERO_DOCUMENTO=b.dni COLLATE Modern_Spanish_CI_AS 
where b.ESTADO='1.-VALIDADA';

UPDATE a SET a.FECHA_LLAMADA =convert(varchar,CONVERT(date,b.FECHA)),
a.DIA=substring(convert(varchar,b.FECHA),9,2)
FROM #FunnelDinersTc a, #VentasDinersTc b 
WHERE A.NUMERO_DOCUMENTO =B.dni COLLATE Modern_Spanish_CI_AS;

update a set a.[NUM_DIA_HABIL]=b.[NUM_DIA_HABIL], a.Semana_Mes=b.Semana_Mes 
from #FunnelDinersTc a 
inner join [ADM_OBJ_TG].[synDiaHabil] b on a.fecha_llamada=b.Fecha

update a 
set a.CNT_LLAMADAS= b.CNT_LLAMADAS
from #FunnelDinersTc a inner join #LlamadasDinersTc1 b 
on a.NUMERO_DOCUMENTO=b.DNI COLLATE Modern_Spanish_CI_AS

/*============================================================================================================================================================*/
/*============================================================================================================================================================*/
/*
ALTER TABLE MAEBA.ADM_OBJ_TG.tGestionMesDinersTc
ALTER COLUMN RHFC NVARCHAR(50) NULL;

ALTER TABLE MAEBA.ADM_OBJ_TG.tGestionMesDinersTc
ADD
    RHFC           NVARCHAR(30) NULL
*/
EXEC sp_rename 
    'ADM_OBJ_TG.tGestionMesEfectiva.MARCA',
    'PERFIL',
    'COLUMN';
select distinct PERFIL from ADM_OBJ_TG.tGestionMesDinersTc
PERFIL as MARCA,
a.GRUPO_EJECUCION as MARCA2,


delete from ADM_OBJ_TG.tGestionMesDinersTc where tMesGestion=DATENAME(MONTH,@inicio) and AÑO_DURACION_BASE=YEAR(@inicio);
insert into ADM_OBJ_TG.tGestionMesDinersTc(
NUMERO_DOCUMENTO,NRG,Dni,TIPO_GESTION,GESTION,SUBGESTION,TIPO,RECORRIDO,
CET,CONT_GEN,Hora_Llamada,Mejor_Telefono,TELEFONO,FECHA_LLAMADA,DIA,
Ejecutivo,RHFC,SUPERVISOR,segundos,VAL,AGENDADOS,[SOLO FH1],[SOLO FH2],
[SOLO FH3],[DOBLE FH],[TRIPLE FH],NUMXDNI,Estado,CntEstado,MntOferta,
CNTVTAS,NUM_DIA_HABIL,Semana_Mes,CNT_LLAMADAS,
AÑO_DURACION_BASE,MES_DURACION_BASE,FECHA_ENVIO,SERVICIO,RETIRO,LINEA_DINERS,
Prioridad,RECENCIA,FLG_REP_MES_ANT,FLG_REP_MESES_ANT,FLG_SIN_ENR,tMesGestion,PROVINCIA,
PERFIL
PROVINCIA,llave,
RANGO_OFERTA,RANGO_TASA,EDAD,DESCRIPCION,
REGION,LOTE)
select a.*,b.AÑO_DURACION_BASE,b.MES_DURACION_BASE,FECHA_ENVIO,b.SERVICIO,b.RETIRO,
cast(REPLACE(REPLACE(b.[LINEA CREDITO DOLARES],',',''),',','') as numeric(18,2) ) as LINEA_DINERS,
b.PROB_CONTACTO as Prioridad,
b.RECURRENCIA AS RECENCIA,
FLG_REP_MES_ANT=CASE WHEN ISNULL(REP1,0)=1 THEN 'Stock' Else 'Nuevo' END ,
FLG_REP_MESES_ANT=CASE WHEN ISNULL(REP2,0)=1 THEN 'Stock' Else 'Nuevo' END, 
FLG_SIN_ENR =CASE WHEN ISNULL(FLAT2,0) =0 THEN 'SIN ENRIQUECER' END,
DATENAME(MONTH,GETDATE()) AS tMesGestion,
b.PROVINCIA,
b.NUMERO_DOCUMENTO+' '+convert(varchar,fecha_envio) llave,
CASE WHEN (b.REP1=1 OR b.REP2=1) THEN 'Stock' ELSE 'Nuevo' END AS CONJUNTO3MESES,
'REGISTROS CARGADOS'  AS 'DESCRIPCION2',
RANGO_OFERTA= CASE 
				WHEN  b.[LINEA CREDITO DOLARES] is null THEN 'A. Sin Datos'
				WHEN cast(REPLACE(b.[LINEA CREDITO DOLARES],',','') as numeric(18,2) )<2000  THEN 'B. 1,000 A 2,000'
				WHEN cast(REPLACE(b.[LINEA CREDITO DOLARES],',','') as numeric(18,2) )<4000  THEN 'C. 2,000 A 4,000'
				WHEN cast(REPLACE(b.[LINEA CREDITO DOLARES],',','') as numeric(18,2) )<6000 THEN 'D. 4,000 A 6,000' 
				WHEN cast(REPLACE(b.[LINEA CREDITO DOLARES],',','') as numeric(18,2) )<8000 THEN 'D. 6,000 A 8,000' 
				WHEN cast(REPLACE(b.[LINEA CREDITO DOLARES],',','') as numeric(18,2) )<10000 THEN 'E. 8,000 A 10,000' 
				WHEN cast(REPLACE(b.[LINEA CREDITO DOLARES],',','') as numeric(18,2) )<12000 THEN 'F. 10,000 A 12,000' 
				else 'G. 12,000 A MÁS' END,
RANGO_TASA = CASE
			WHEN convert(float,left(b.[TASA(TEA)],2))>=14 AND convert(float,left(b.[TASA(TEA)],2))<=30 THEN '2.14 A 30%' 
			WHEN convert(float,left(b.[TASA(TEA)],2))> 30 AND convert(float,left(b.[TASA(TEA)],2))<=40 THEN '3.30 A 40%' 
			WHEN convert(float,left(b.[TASA(TEA)],2))> 40 AND convert(float,left(b.[TASA(TEA)],2))<=50 THEN '4.40 A 50%' 
			WHEN convert(float,left(b.[TASA(TEA)],2))> 50 AND convert(float,left(b.[TASA(TEA)],2))<=60 THEN '5.50 A 60%' 
			WHEN convert(float,left(b.[TASA(TEA)],2))> 60 AND convert(float,left(b.[TASA(TEA)],2))<=80 THEN '6.60 A 70%'
			ELSE '1.MENOS A 14%' 
			end,
EDAD = CASE
			WHEN b.FEC_NAC is null THEN '0.Sin Dato'
			WHEN convert(int,DATEDIFF(YEAR, b.FEC_NAC, GETDATE()))>=20 AND convert(int,DATEDIFF(YEAR, b.FEC_NAC, GETDATE()))<=30 THEN '1.20 A 30'
			WHEN convert(int,DATEDIFF(YEAR, b.FEC_NAC, GETDATE()))>=30 AND convert(int,DATEDIFF(YEAR, b.FEC_NAC, GETDATE()))<=40 THEN '2.30 A 40'
			WHEN convert(int,DATEDIFF(YEAR, b.FEC_NAC, GETDATE()))>=40 AND convert(int,DATEDIFF(YEAR, b.FEC_NAC, GETDATE()))<=50 THEN '3.40 A 50'
			WHEN convert(int,DATEDIFF(YEAR, b.FEC_NAC, GETDATE()))>=50 AND convert(int,DATEDIFF(YEAR, b.FEC_NAC, GETDATE()))<=60 THEN '4.50 A 60'
			WHEN convert(int,DATEDIFF(YEAR, b.FEC_NAC, GETDATE()))>=60 AND convert(int,DATEDIFF(YEAR, b.FEC_NAC, GETDATE()))<=75 THEN '5.60 A 70'
			ELSE '6.70 a más' end,
'DESCRIPCION'=CASE WHEN
(CONVERT(float,REPLACE(b.L_IBK,',','.'))>CONVERT(float,REPLACE(b.L_BCP,',','.'))
AND CONVERT(float,REPLACE(b.L_IBK,',','.'))>CONVERT(float, REPLACE(b.L_BBVA  ,',','.'))
 AND CONVERT(float,REPLACE(b.L_IBK,',','.'))>CONVERT(float,REPLACE(b.L_SCO   ,',','.'))
 AND CONVERT(float,REPLACE(b.L_IBK,',','.'))>CONVERT(float,REPLACE(b.L_BIF   ,',','.') )
 AND CONVERT(float,REPLACE(b.L_IBK,',','.'))>CONVERT(float,REPLACE(b.L_CITI  ,',','.') )
 AND CONVERT(float,REPLACE(b.L_IBK,',','.'))>CONVERT(float,REPLACE(b.L_FIN   ,',','.') )
 AND CONVERT(float,REPLACE(b.L_IBK,',','.'))>CONVERT(float,REPLACE(b.L_RIP   ,',','.') )
 AND CONVERT(float,REPLACE(b.L_IBK,',','.'))>CONVERT(float,REPLACE(b.L_CMR   ,',','.') )
 AND CONVERT(float,REPLACE(b.L_IBK,',','.'))>CONVERT(float,REPLACE(b.L_CRESCO,',','.')    )
 AND CONVERT(float,REPLACE(b.L_IBK,',','.'))>CONVERT(float,REPLACE(b.L_CEN   ,',','.')  )
 AND CONVERT(float,REPLACE(b.L_IBK,',','.'))>CONVERT(float,REPLACE(b.L_AZT   ,',','.')   )
 AND CONVERT(float,REPLACE(b.L_IBK,',','.'))>CONVERT(float,REPLACE(b.L_UNO   ,',','.') )
 AND CONVERT(float,REPLACE(b.L_IBK,',','.'))>CONVERT(float,REPLACE(b.L_GNB   ,',','.')   )
 AND CONVERT(float,REPLACE(b.L_IBK,',','.'))>CONVERT(float,REPLACE(b.L_EFE   ,',','.')  )
 AND CONVERT(float,REPLACE(b.L_IBK,',','.'))>CONVERT(float,REPLACE(b.L_COM   ,',','.')  )
 AND CONVERT(float,REPLACE(b.L_IBK,',','.'))>CONVERT(float,REPLACE(b.L_NAC   ,',','.')  )) THEN 'IBK'
 WHEN
(CONVERT(float,REPLACE(b.L_BCP,',','.'))>CONVERT(float,REPLACE(b.L_IBK,',','.'))
AND CONVERT(float,REPLACE(b.L_BCP,',','.'))>CONVERT(float, REPLACE(b.L_BBVA  ,',','.'))
 AND CONVERT(float,REPLACE(b.L_BCP,',','.'))>CONVERT(float,REPLACE(b.L_SCO   ,',','.'))
 AND CONVERT(float,REPLACE(b.L_BCP,',','.'))>CONVERT(float,REPLACE(b.L_BIF   ,',','.') )
 AND CONVERT(float,REPLACE(b.L_BCP,',','.'))>CONVERT(float,REPLACE(b.L_CITI  ,',','.') )
 AND CONVERT(float,REPLACE(b.L_BCP,',','.'))>CONVERT(float,REPLACE(b.L_FIN   ,',','.') )
 AND CONVERT(float,REPLACE(b.L_BCP,',','.'))>CONVERT(float,REPLACE(b.L_RIP   ,',','.') )
 AND CONVERT(float,REPLACE(b.L_BCP,',','.'))>CONVERT(float,REPLACE(b.L_CMR   ,',','.') )
 AND CONVERT(float,REPLACE(b.L_BCP,',','.'))>CONVERT(float,REPLACE(b.L_CRESCO,',','.')    )
 AND CONVERT(float,REPLACE(b.L_BCP,',','.'))>CONVERT(float,REPLACE(b.L_CEN   ,',','.')  )
 AND CONVERT(float,REPLACE(b.L_BCP,',','.'))>CONVERT(float,REPLACE(b.L_AZT   ,',','.')   )
 AND CONVERT(float,REPLACE(b.L_BCP,',','.'))>CONVERT(float,REPLACE(b.L_UNO   ,',','.') )
 AND CONVERT(float,REPLACE(b.L_BCP,',','.'))>CONVERT(float,REPLACE(b.L_GNB   ,',','.')   )
 AND CONVERT(float,REPLACE(b.L_BCP,',','.'))>CONVERT(float,REPLACE(b.L_EFE   ,',','.')  )
 AND CONVERT(float,REPLACE(b.L_BCP,',','.'))>CONVERT(float,REPLACE(b.L_COM   ,',','.')  )
 AND CONVERT(float,REPLACE(b.L_BCP,',','.'))>CONVERT(float,REPLACE(b.L_NAC   ,',','.')  )) THEN 'L_BCP'

 WHEN
(CONVERT(float,REPLACE(b.L_BBVA,',','.'))>CONVERT(float,REPLACE(b.L_IBK,',','.'))
AND CONVERT(float,REPLACE(b.L_BBVA,',','.'))>CONVERT(float, REPLACE(b.L_BCP  ,',','.'))
 AND CONVERT(float,REPLACE(b.L_BBVA,',','.'))>CONVERT(float,REPLACE(b.L_SCO   ,',','.'))
 AND CONVERT(float,REPLACE(b.L_BBVA,',','.'))>CONVERT(float,REPLACE(b.L_BIF   ,',','.') )
 AND CONVERT(float,REPLACE(b.L_BBVA,',','.'))>CONVERT(float,REPLACE(b.L_CITI  ,',','.') )
 AND CONVERT(float,REPLACE(b.L_BBVA,',','.'))>CONVERT(float,REPLACE(b.L_FIN   ,',','.') )
 AND CONVERT(float,REPLACE(b.L_BBVA,',','.'))>CONVERT(float,REPLACE(b.L_RIP   ,',','.') )
 AND CONVERT(float,REPLACE(b.L_BBVA,',','.'))>CONVERT(float,REPLACE(b.L_CMR   ,',','.') )
 AND CONVERT(float,REPLACE(b.L_BBVA,',','.'))>CONVERT(float,REPLACE(b.L_CRESCO,',','.')    )
 AND CONVERT(float,REPLACE(b.L_BBVA,',','.'))>CONVERT(float,REPLACE(b.L_CEN   ,',','.')  )
 AND CONVERT(float,REPLACE(b.L_BBVA,',','.'))>CONVERT(float,REPLACE(b.L_AZT   ,',','.')   )
 AND CONVERT(float,REPLACE(b.L_BBVA,',','.'))>CONVERT(float,REPLACE(b.L_UNO   ,',','.') )
 AND CONVERT(float,REPLACE(b.L_BBVA,',','.'))>CONVERT(float,REPLACE(b.L_GNB   ,',','.')   )
 AND CONVERT(float,REPLACE(b.L_BBVA,',','.'))>CONVERT(float,REPLACE(b.L_EFE   ,',','.')  )
 AND CONVERT(float,REPLACE(b.L_BBVA,',','.'))>CONVERT(float,REPLACE(b.L_COM   ,',','.')  )
 AND CONVERT(float,REPLACE(b.L_BBVA,',','.'))>CONVERT(float,REPLACE(b.L_NAC   ,',','.')  )) THEN 'L_BBVA'

 WHEN
(CONVERT(float,REPLACE(b.L_SCO,',','.'))>CONVERT(float,REPLACE(b.L_IBK,',','.'))
AND CONVERT(float,REPLACE(b.L_SCO,',','.'))>CONVERT(float, REPLACE(b.L_BCP  ,',','.'))
 AND CONVERT(float,REPLACE(b.L_SCO,',','.'))>CONVERT(float,REPLACE(b.L_BBVA   ,',','.'))
 AND CONVERT(float,REPLACE(b.L_SCO,',','.'))>CONVERT(float,REPLACE(b.L_BIF   ,',','.') )
 AND CONVERT(float,REPLACE(b.L_SCO,',','.'))>CONVERT(float,REPLACE(b.L_CITI  ,',','.') )
 AND CONVERT(float,REPLACE(b.L_SCO,',','.'))>CONVERT(float,REPLACE(b.L_FIN   ,',','.') )
 AND CONVERT(float,REPLACE(b.L_SCO,',','.'))>CONVERT(float,REPLACE(b.L_RIP   ,',','.') )
 AND CONVERT(float,REPLACE(b.L_SCO,',','.'))>CONVERT(float,REPLACE(b.L_CMR   ,',','.') )
 AND CONVERT(float,REPLACE(b.L_SCO,',','.'))>CONVERT(float,REPLACE(b.L_CRESCO,',','.')    )
 AND CONVERT(float,REPLACE(b.L_SCO,',','.'))>CONVERT(float,REPLACE(b.L_CEN   ,',','.')  )
 AND CONVERT(float,REPLACE(b.L_SCO,',','.'))>CONVERT(float,REPLACE(b.L_AZT   ,',','.')   )
 AND CONVERT(float,REPLACE(b.L_SCO,',','.'))>CONVERT(float,REPLACE(b.L_UNO   ,',','.') )
 AND CONVERT(float,REPLACE(b.L_SCO,',','.'))>CONVERT(float,REPLACE(b.L_GNB   ,',','.')   )
 AND CONVERT(float,REPLACE(b.L_SCO,',','.'))>CONVERT(float,REPLACE(b.L_EFE   ,',','.')  )
 AND CONVERT(float,REPLACE(b.L_SCO,',','.'))>CONVERT(float,REPLACE(b.L_COM   ,',','.')  )
 AND CONVERT(float,REPLACE(b.L_SCO,',','.'))>CONVERT(float,REPLACE(b.L_NAC   ,',','.')  )) THEN 'L_SCO'
 ELSE 'OTROS' END,
 REGION=CASE	WHEN ISNULL(A.[PROVINCIA],'') ='' THEN 'Leads sin región'
			WHEN A.[PROVINCIA] ='LIMA' OR A.[PROVINCIA] ='CALLAO' THEN 'Leads Lima'
			WHEN A.[PROVINCIA] <>'LIMA' THEN 'Leads Provincia' end,
LOTE=CASE	WHEN b.FECHA_ENVIO ='2026-07-01' THEN 'ENVIO 01'
		else 'OTROS'
		end
from #FunnelDinersTc a
inner join [ADM_OBJ_TG].[synBaseMaestraDinersTc] b
on a.NUMERO_DOCUMENTO=b.NUMERO_DOCUMENTO












































DROP TABLE IF EXISTS
    adm_obj_tg.RhfDinersTc,
    adm_obj_tg.NumxDnixDinersTc,
    adm_obj_tg.FunnelDinersTc,
    adm_obj_tg.LlamadasDinersTc1,
    adm_obj_tg.LlamadasDinersTc2,
    adm_obj_tg.FunnelResumenDinersTc,
    adm_obj_tg.FunnelDiasUtilesDinersTc,
    adm_obj_tg.tmp_paraasesorescpordia20250215,
    adm_obj_tg.synUsuariosTg_temp,
    adm_obj_tg.#temp_FunnelDinersTc;

/*============================================================================================================================================================*/
/*============================================================================================================================================================*/
-- solo se esta agregando una tabla que reemplazara momentaneamente a synUsuariosTg_temp :/ hasta que ocurra el milagro y la info este actualizada

SELECT
    A.tNivel1,
    A.tDocumentoVici,
    A.tSupervisor,
    A.tNombreCompleto,
    A.tEstado,
    A.tCondicion,
    A.tInicio,
    A.tCese,
    A.tModalidad,
    ISNULL(B.tsede, 'SIN INFORMACION') AS tsede
INTO [ADM_OBJ_TG].synUsuariosTg_temp
FROM [ADM_OBJ_TG].synUsuariosTg A
LEFT JOIN ODIN.dbo.borrar_info_agentes B
ON A.tDocumentoVici = B.tDocumentoVici

INSERT INTO [ADM_OBJ_TG].synUsuariosTg_temp
SELECT
    B.tNivel1,
    B.tDocumentoVici,
    B.tSupervisor,
    B.tNombreCompleto,
    B.tEstado,
    B.tCondicion,
    B.tInicio,
    null as tcese,
    B.tModalidad,
    B.tsede
FROM ODIN.dbo.borrar_info_agentes B
WHERE NOT EXISTS (
    SELECT 1
    FROM [ADM_OBJ_TG].synUsuariosTg A
    WHERE A.tDocumentoVici = B.tDocumentoVici
)

update a
set a.tSupervisor=b.tSupervisor  -- parece que supervisor no esta actualizado
FROM [ADM_OBJ_TG].synUsuariosTg_temp A
inner join ODIN.dbo.borrar_info_agentes B
on a.tDocumentoVici=b.tDocumentoVici
DECLARE @fecha_actual DATE = CAST(GETDATE() AS DATE) -- esto ayuda a obtener el valor de antiguedad

/*============================================================================================================================================================*/
/*============================================================================================================================================================*/




CREATE TABLE adm_obj_tg.RhfDinersTc (RHF VARCHAR(3),[SOLO FH1]INT,[SOLO FH2]INT,[SOLO FH3]INT,[DOBLE FH]INT,[TRIPLE FH] INT)
INSERT INTO adm_obj_tg.RhfDinersTc
VALUES  ('000',0,0,0,0,0),('100',1,0,0,0,0),('020',0,1,0,0,0),('003',0,0,1,0,0),('120',0,0,0,1,0),('103',0,0,0,1,0),('023',0,0,0,1,0),('123',0,0,0,0,1)
/*============================================================================================================================================================*/
/*============================================================================================================================================================*/
select NUMERO_DOCUMENTO as DNI,count(*) as numxdni into adm_obj_tg.NumxDnixDinersTc from [ODIN].[dbo].[tNumeroDinersTc] group by NUMERO_DOCUMENTO
create index NUMERO_DOCUMENTO on adm_obj_tg.NumxDnixDinersTc(DNI)
/*============================================================================================================================================================*/
/*============================================================================================================================================================*/
SELECT 
A.NUMERO_DOCUMENTO,
CASE	
		WHEN a.FECHA_ENVIO='2026-07-01'		THEN 'ENVIO_1'
	 END AS FECHA_ENVIO,
A.SERVICIO,
A.AÑO_DURACION_BASE,
A.MES_DURACION_BASE,
A.RETIRO,
cast(REPLACE(REPLACE(a.[LINEA CREDITO DOLARES],',',''),',','') as numeric(18,2) ) as LINEA_DINERS,
REGION=CASE	WHEN ISNULL(A.[PROVINCIA],'') ='' THEN 'Leads sin región'
			WHEN A.[PROVINCIA] ='LIMA' OR A.[PROVINCIA] ='CALLAO' THEN 'Leads Lima'
			WHEN A.[PROVINCIA] <>'LIMA' THEN 'Leads Provincia' end,
RANGO_OFERTA= CASE 
RANGO_TASA = CASE
EDAD = CASE
isnull(A.PROB_CONTACTO,a.GRUPO_EJECUCION) as Prioridad,
PERFIL as MARCA,
a.GRUPO_EJECUCION as MARCA2,
A.RECURRENCIA AS PERFIL,
a.MARCA as Nombre_Campana,
CASE 
 'DESCRIPCION',
FLG_REP_MES_ANT=CASE WHEN ISNULL(REP1,0)=1 THEN 'Stock' Else 'Nuevo' END ,
FLG_REP_MESES_ANT=CASE WHEN ISNULL(REP2,0)=1 THEN 'Stock' Else 'Nuevo' END, 
DATENAME(MONTH,GETDATE()) AS tMesGestion,
FLG_SIN_ENR =CASE WHEN ISNULL(FLAT2,0) =0 THEN 'SIN ENRIQUECER' END,
a.PROVINCIA,
NUMERO_DOCUMENTO+' '+convert(varchar,fecha_envio) llave,
into adm_obj_tg.FunnelDinersTc
FROM [ADM_OBJ_TG].[synBaseMaestraDinersTc] a /*select *, '' as leadslanding from [192.168.2.30].[Dantalion].[dbo].[dinerstc_aux_20241016_sinrep] UNION ALL  select '' as fechaenvio, * from */
LEFT JOIN [ADM_OBJ_TG].[synTmpLLamadasDinersTc] b on a.NUMERO_DOCUMENTO=b.Dni
LEFT JOIN adm_obj_tg.NumxDnixDinersTc C ON a.NUMERO_DOCUMENTO=c.DNI
--cambie a synUsuariosTg por synUsuariosTg_temp
LEFT JOIN [ADM_OBJ_TG].synUsuariosTg_temp d ON b.DNI_EJECUTIVO=d.tNivel1 COLLATE Modern_Spanish_CI_AS
LEFT JOIN adm_obj_tg.RhfDinersTc E1 ON b.RHFC=E1.RHF COLLATE Modern_Spanish_CI_AS  --WHERE a.FECHA_ENVIO LIKE '2024-02%'

--select distinct([TASA(TEA)]) from [ADM_OBJ_TG].[synBaseMaestraDinersTc] 
/*============================================================================================================================================================*/

--SELECT top(10)* FROM [ADM_OBJ_TG].[synTmpLLamadasDinersTc]

/*============================================================================================================================================================*/
update a 
set a.TIPO_GESTION  ='CONTACTO EFECTIVO', 
	a.GESTION ='NO ESCUCHO OFERTA', 
	a.SUBGESTION ='CLIENTE CORTO SIN ESCUCHAR OFERTA'
from adm_obj_tg.FunnelDinersTc a left join [ADM_OBJ_TG].[synVentasDinersTc] b on a.NUMERO_DOCUMENTO=b.dni
where b.dni is null and a.SUBGESTION='CLIENTE ACEPTO OFERTA TC'
UPDATE a
SET a.TIPO_GESTION='CONTACTO EFECTIVO',
	a.GESTION ='ACEPTA CAMPANA',
	a.SUBGESTION='CLIENTE ACEPTO OFERTA TC'
from adm_obj_tg.FunnelDinersTc a inner join [ADM_OBJ_TG].[synVentasDinersTc] b on a.NUMERO_DOCUMENTO=b.dni
/*============================================================================================================================================================*/
/*============================================================================================================================================================*/
UPDATE a SET a.Ejecutivo=isnull(c.tNombreCompleto,b.EJECUTIVO)
FROM adm_obj_tg.FunnelDinersTc a 
inner join [ADM_OBJ_TG].[synVentasDinersTc] b on a.NUMERO_DOCUMENTO=b.dni
--cambie synUsuariosTg por synUsuariosTg_temp
inner join [ADM_OBJ_TG].synUsuariosTg_temp c on b.dni=c.tDocumentoVici
UPDATE  a SET a.MntOferta =convert(int,replace(REPLACE(REPLACE(MONTO,'.',''),',',''),'	',''))
FROM adm_obj_tg.FunnelDinersTc a inner join [ADM_OBJ_TG].[synVentasDinersTc] b on a.NUMERO_DOCUMENTO=b.dni and substring(convert(varchar,a.llave),10,10) <= convert(varchar,b.FECHA);
UPDATE  a SET a.CNTVTAS =1
FROM adm_obj_tg.FunnelDinersTc a inner join [ADM_OBJ_TG].[synVentasDinersTc] b on a.NUMERO_DOCUMENTO=b.dni and substring(convert(varchar,a.llave),10,10) <= convert(varchar,b.FECHA);
UPDATE  a SET a.CNTVTAS =1
FROM adm_obj_tg.FunnelDinersTc a inner join [ADM_OBJ_TG].[synVentasDinersTc] b on a.NUMERO_DOCUMENTO=b.dni COLLATE Modern_Spanish_CI_AS and substring(convert(varchar,a.llave),10,10) <= convert(varchar,b.FECHA); 
UPDATE a SET a.Estado=isnull(replace(b.ESTADO,'-','EN PROCESO'),'EN PROCESO')
FROM adm_obj_tg.FunnelDinersTc a inner join [ADM_OBJ_TG].[synVentasDinersTc] b on a.NUMERO_DOCUMENTO=b.dni and substring(convert(varchar,a.llave),10,10) <= convert(varchar,b.FECHA);
UPDATE a SET a.CntEstado=1
FROM adm_obj_tg.FunnelDinersTc a inner join [ADM_OBJ_TG].[synVentasDinersTc] b on a.NUMERO_DOCUMENTO=b.dni where b.ESTADO='1.-VALIDADA' and substring(convert(varchar,a.llave),10,10) <= convert(varchar,b.FECHA);
UPDATE a SET a.CntEstado=1
FROM adm_obj_tg.FunnelDinersTc a inner join [ADM_OBJ_TG].[synVentasDinersTc] b on a.NUMERO_DOCUMENTO=b.dni COLLATE Modern_Spanish_CI_AS WHERE b.ESTADO='1.-VALIDADA' and substring(convert(varchar,a.llave),10,10) <= convert(varchar,b.FECHA); 

UPDATE a SET a.FECHA_LLAMADA =convert(varchar,CONVERT(date,b.FECHA)), a.DIA=substring(convert(varchar,b.FECHA),9,2)
FROM adm_obj_tg.FunnelDinersTc a, [ADM_OBJ_TG].[synVentasDinersTc] b WHERE A.NUMERO_DOCUMENTO =B.dni;

update adm_obj_tg.FunnelDinersTc set GESTION='CLIENTE INTERESADO', SUBGESTION='CLIENTE INTERESADO - NO DESEA QUE LE ENVIEN INFORMACIÓN'
where SUBGESTION in ('DESEA QUE LO CONTACTEN - WHATSAPP','DESEA QUE LO CONTACTEN - WHATSAPP O CORREO','CLIENTE INTERESADO - WHATSAPP','CLIENTE INTERESADO - CORREO','CLIENTE INTERESADO - WHATSAPP Y CORREO');
/*============================================================================================================================================================*/
--select * from  [ADM_OBJ_TG].[synVentasDinersTc]
/*============================================================================================================================================================*/
update a set a.[NUM_DIA_HABIL]=b.[NUM_DIA_HABIL], a.Semana_Mes=b.Semana_Mes from adm_obj_tg.FunnelDinersTc a inner join [ADM_OBJ_TG].[synDiaHabil] b on a.fecha_llamada=b.Fecha
/*============================================================================================================================================================*/
/*============================================================================================================================================================*/
SELECT DNI AS DNI_LLAMADAS, COUNT(*) AS CNT_LLAMADAS1 
INTO adm_obj_tg.LlamadasDinersTc1 FROM [ADM_OBJ_TG].[synTmpLlamadasGeneralDinersTc] GROUP BY DNI
/*============================================================================================================================================================*/
/*============================================================================================================================================================*/
/*
ALTER TABLE [ADM_OBJ_TG].[tGestionMesDinersTc]
ADD tsede VARCHAR(50),
	tModalidad VARCHAR(50),
	tCondicion VARCHAR(50),
	tEstado VARCHAR(50),
    tantiguedad VARCHAR(50);
*/

/*============================================================================================================================================================*/
/*============================================================================================================================================================*/
select a.*,b.CNT_LLAMADAS1 as CNT_LLAMADAS,CASE WHEN c.FLAT3=1 THEN 1 ELSE 0 END AS SOCIO,
CASE WHEN a.TIPO_GESTION in ('CONTACTO NO EFECTIVO') THEN 1 ELSE 0 END NO_CET,C.N_BASE,c.MARCA3 as segmentacion,
CASE WHEN a.CNTVTAS=1 then 1 when a.FECHA_ENVIO='ENVIO_3' then 1 else 0 end as neg,c.MARCA as mac,
usu.tsede,
usu.tModalidad,
usu.tCondicion,
usu.tEstado,
tantiguedad =
    CASE 
        WHEN usu.tInicio is null then 'NO APLICA'
        WHEN 
			DAY(usu.tInicio) <>1 and
			DATEADD(MONTH, 1, DATEADD(DAY, 1, EOMONTH(usu.tInicio)))>@fecha_actual
		then 'NUEVO'
        WHEN 
			DATEADD(DAY, 1, EOMONTH(usu.tInicio)) >@fecha_actual
		then 'NUEVO'
		ELSE 
		'ANTIGUO'
	END
into #temp_FunnelDinersTc
from adm_obj_tg.FunnelDinersTc a left join adm_obj_tg.LlamadasDinersTc1 b on a.NUMERO_DOCUMENTO=b.DNI_LLAMADAS
left join [ADM_OBJ_TG].[synBaseMaestraDinersTc] c 
on a.NUMERO_DOCUMENTO=c.NUMERO_DOCUMENTO and a.llave=c.NUMERO_DOCUMENTO+' '+convert(varchar,c.fecha_envio)
left JOIN [ADM_OBJ_TG].synUsuariosTg_temp usu ON a.DNI_EJECUTIVO = usu.tNivel1

ALTER TABLE #temp_FunnelDinersTc
DROP COLUMN DNI_EJECUTIVO;

UPDATE  a 
SET a.CNTVTAS =1
FROM #temp_FunnelDinersTc a 
inner join [ADM_OBJ_TG].[synVentasDinersTc] b 
on a.NUMERO_DOCUMENTO=b.dni 
AND A.CNTVTAS IS NULL
AND FECHA>='2026-06-01'


delete from [ADM_OBJ_TG].[tGestionMesDinersTc] where tMesGestion=DATENAME(MONTH,GETDATE())/*'Mayo'*/;
insert into [ADM_OBJ_TG].[tGestionMesDinersTc]
select * from #temp_FunnelDinersTc

UPDATE [ADM_OBJ_TG].[tGestionMesDinersTc] SET RETIRO = ''
where tMesGestion=DATENAME(MONTH,GETDATE()) AND AÑO_DURACION_BASE = DATENAME(YEAR,GETDATE()) 
AND CNTVTAS = '1'



--select a.*,b.CNT_LLAMADAS1 as CNT_LLAMADAS from adm_obj_tg.FunnelDinersTc a left join adm_obj_tg.LlamadasDinersTc1 b on a.NUMERO_DOCUMENTO=b.DNI_LLAMADAS


/*============================================================================================================================================================*/
/*============================================================================================================================================================*/
delete from [ADM_OBJ_TG].[ResumenDinersTc] where tMesGestion=DATENAME(MONTH,GETDATE());
insert into [ADM_OBJ_TG].[ResumenDinersTc]
select a.* from (
select tMesGestion,NUMERO_DOCUMENTO+' '+convert(varchar,fecha_envio) llave,'Entregados' MedicionDatos,sum(NRG) Q from [ADM_OBJ_TG].[tGestionMesDinersTc] where NRG=1 GROUP BY NUMERO_DOCUMENTO,FECHA_ENVIO,tMesGestion
UNION ALL
select tMesGestion,NUMERO_DOCUMENTO+' '+convert(varchar,fecha_envio) llave,'Recorridos' MedicionDatos,sum(RECORRIDO) Q from [ADM_OBJ_TG].[tGestionMesDinersTc] where RECORRIDO=1 GROUP BY NUMERO_DOCUMENTO,FECHA_ENVIO,tMesGestion
UNION ALL
select tMesGestion,NUMERO_DOCUMENTO+' '+convert(varchar,fecha_envio) llave,'Llamados' MedicionDatos,sum(CNT_LLAMADAS) Q from [ADM_OBJ_TG].[tGestionMesDinersTc] where RECORRIDO=1 GROUP BY NUMERO_DOCUMENTO,FECHA_ENVIO,tMesGestion
UNION ALL
select tMesGestion,NUMERO_DOCUMENTO+' '+convert(varchar,fecha_envio) llave,'Titular' MedicionDatos,sum(CET) Q from [ADM_OBJ_TG].[tGestionMesDinersTc] where CET=1 GROUP BY NUMERO_DOCUMENTO,FECHA_ENVIO,tMesGestion
UNION ALL
select tMesGestion,NUMERO_DOCUMENTO+' '+convert(varchar,fecha_envio) llave,'Ventas' MedicionDatos,sum(CNTVTAS) Q from [ADM_OBJ_TG].[tGestionMesDinersTc] where CNTVTAS=1 GROUP BY NUMERO_DOCUMENTO,FECHA_ENVIO,tMesGestion ) a where tMesGestion=DATENAME(MONTH,GETDATE());
/*============================================================================================================================================================*/
/*============================================================================================================================================================*/
--select 'Efectividad BD' MedicionDatos,cast(sum(CNTVTAS)/sum(NRG) as float) Q from [ADM_OBJ_TG].[tGestionMesDinersTc] where CNTVTAS=1 GROUP BY NUMERO_DOCUMENTO,FECHA_ENVIO,tMesGestion


/*============================================================================================================================================================*/
If object_id('[ADM_OBJ_TG].VentasDinersTcFunnel') is not null
Drop table [ADM_OBJ_TG].VentasDinersTcFunnel;

DECLARE @MES1 VARCHAR(100)
DECLARE @MES2 VARCHAR(100)
DECLARE @MES3 VARCHAR(100)

SET @MES1 = DATENAME(MONTH, DATEADD(MM,0,GETDATE()))
SET @MES2 = DATENAME(MONTH, DATEADD(MM,-1,GETDATE()))
SET @MES3 = DATENAME(MONTH, DATEADD(MM,-2,GETDATE()))

select vnt.*,cast(dt.NUM_DIA_HABIL as int) dia_,dt.[Semana_Mes],tp.[tSupervisor]
into [ADM_OBJ_TG].VentasDinersTcFunnel
from [ADM_OBJ_TG].[synVentasDinersTcFunnel] VNT INNER JOIN [ADM_OBJ_TG].[synDiaHabil] DT ON VNT.FECHA=DT.Fecha
--cambie synUsuariosTg por synUsuariosTg_temp
inner join [ADM_OBJ_TG].synUsuariosTg_temp tp on vnt.DNIEjecutivo=tp.tDocumentoVici
where DATENAME(MONTH,VNT.FECHA) in (@MES1,@MES2,@MES3) and CAMPANA='DinersTc'
--AND ANIO=2024
order by FECHA

/*============================================================================================================================================================*/
If object_id('ADM_OBJ_TG.PassingDinersTcFunnel') is not null
Drop table ADM_OBJ_TG.PassingDinersTcFunnel;

SELECT fecha,MES_VENTA,dia,EJECUTIVO,ESTADO,count(*) registros 
into ADM_OBJ_TG.PassingDinersTcFunnel
FROM ADM_OBJ_TG.VentasDinersTcFunnel
group by fecha,MES_VENTA,dia,EJECUTIVO,ESTADO

/*============================================================================================================================================================*/

SELECT A.*,B.DIAS_UTILES 
INTO ADM_OBJ_TG.FunnelDiasUtilesDinersTc
FROM
(SELECT DATENAME(MONTH, DATEADD(MM,0,GETDATE())) MES,MAX(NUM_DIA_HABIL) DIA_HABIL 
FROM [MAEBA].[ADM_OBJ_TG].[tGestionMesDinersTc] WHERE tMesGestion=DATENAME(MONTH, DATEADD(MM,0,GETDATE()))
AND FECHA_LLAMADA>=DATEADD(DAY, 1, EOMONTH(GETDATE(), - 1))) A
INNER JOIN
(SELECT DATENAME(MONTH, DATEADD(MM,0,GETDATE())) MES,COUNT(*) DIAS_UTILES 
FROM [ADM_OBJ_TG].[synDiaHabil] WHERE Fecha_Txt LIKE (LEFT(DATEADD(DAY, 1, EOMONTH(GETDATE(), - 1)),7)+'%') AND Nombre_dia!='Domingo') B ON A.MES=B.MES

SELECT tMesGestion,SUM(NRG) LEADS,
COUNT([CNTVTAS]) VENTAS,SUM(RECORRIDO) RECORRIDOS,SUM(CET) CET,
cast(SUM(CNT_LLAMADAS) as numeric(10,2)) CNT_LLAMADAS
INTO ADM_OBJ_TG.FunnelResumenDinersTc
FROM [MAEBA].[ADM_OBJ_TG].[tGestionMesDinersTc] AVC 
WHERE tMesGestion=DATENAME(MONTH, DATEADD(MM,0,GETDATE()))
GROUP BY tMesGestion;

If object_id('ADM_OBJ_TG.FunnelResumenDTC') is not null
Drop table ADM_OBJ_TG.FunnelResumenDTC;
SELECT A.*,D.DIA_HABIL,D.DIAS_UTILES,
CAST((VENTAS/DIA_HABIL) AS INT)*DIAS_UTILES AS PROYECCION,
cast(CNT_LLAMADAS/RECORRIDOS as decimal(10,4)) Intensidad
INTO ADM_OBJ_TG.FunnelResumenDTC
FROM ADM_OBJ_TG.FunnelResumenDinersTc A LEFT JOIN ADM_OBJ_TG.FunnelDiasUtilesDinersTc D
ON A.tMesGestion=D.MES;

/*=============REPORTE DE TELEFONOS==========================================================================================*/

IF OBJECT_ID('dbo.tReportdinersColumn','U') IS NOT NULL
    DROP  TABLE dbo.tReportdinersColumn;

EXEC [ATHENA].[ADM_OBJ_TELF].[ReporteTelfDinersTc_V2];

SELECT *
INTO dbo.tReportdinersColumn
FROM [ATHENA].[dbo].[tReportdinersColumn];

--select * from tReportdinersColumn


/*=======================================================================================================*/
/*============================================================================================================================================================*/
If object_id('adm_obj_tg.RhfDinersTc') is not null
Drop table adm_obj_tg.RhfDinersTc;
If object_id('adm_obj_tg.NumxDnixDinersTc') is not null
Drop table adm_obj_tg.NumxDnixDinersTc;
If object_id('adm_obj_tg.FunnelDinersTc') is not null
Drop table adm_obj_tg.FunnelDinersTc;
If object_id('adm_obj_tg.LlamadasDinersTc1') is not null
Drop table adm_obj_tg.LlamadasDinersTc1;
If object_id('adm_obj_tg.LlamadasDinersTc2') is not null
Drop table adm_obj_tg.LlamadasDinersTc2;
If object_id('ADM_OBJ_TG.FunnelResumenDinersTc') is not null
Drop table ADM_OBJ_TG.FunnelResumenDinersTc;
If object_id('ADM_OBJ_TG.FunnelDiasUtilesDinersTc') is not null
Drop table ADM_OBJ_TG.FunnelDiasUtilesDinersTc;

EXEC [POSEIDON].[dbo].[reporte_dinerstc_CUATIRLES_20250121];


If object_id('tmp_paraasesorescpordia20250215') is not null 
Drop table tmp_paraasesorescpordia20250215;

If object_id('TMP_PARAASESORPORDIADE8A15') is not null 
Drop table TMP_PARAASESORPORDIADE8A15;

If object_id('asesorpordiade8a15') is not null 
Drop table asesorpordiade8a15;

SELECT * into tmp_paraasesorescpordia20250215
  FROM [THOTH].[dbo].[Tmp_LLamadas_Diners_TC] where Fecha_Llamada like '2026-02%' and Hora_Llamada>=8 and Hora_Llamada<=15 and DNI_Ejecutivo<>'VDAD' and DNI_Ejecutivo<>'42845139' and DNI_Ejecutivo<>'6666'

  select ROW_NUMBER() over( partition by convert(VARCHAR,fecha_llam)+' '+dni_ejecutivo order by Fecha_hora_llamada DESC ) AS ASESORFECHA_LLAMADAORDERFECHAHORALLAMDESC, * INTO TMP_PARAASESORPORDIADE8A15 from tmp_paraasesorescpordia20250215


  select *, DATENAME(MONTH, GETDATE()) AS tMesGestion into asesorpordiade8a15 from TMP_PARAASESORPORDIADE8A15 where ASESORFECHA_LLAMADAORDERFECHAHORALLAMDESC=1
  --select * from asesorpordiade8a15




5

In [ ]:
NUMERO_DOCUMENTO,NRG,Dni,TIPO_GESTION,GESTION,SUBGESTION,TIPO,RECORRIDO,CET,CONT_GEN,Hora_Llamada,Mejor_Telefono,TELEFONO,FECHA_LLAMADA,DIA,Ejecutivo,SUPERVISOR,segundos,VAL,AGENDADOS,DNI_EJECUTIVO,RHFC,SOLO FH1,SOLO FH2,SOLO FH3,DOBLE FH,TRIPLE FH,NUMXDNI,Estado,CntEstado,MntOferta,CNTVTAS,NUM_DIA_HABIL,Semana_Mes,CNT_LLAMADAS,

In [5]:

query = """
    SELECT top(10)* FROM maeba.[ADM_OBJ_TG].[tGestionMesEfectivaN]
  """
df_prueba= pd.read_sql(query,engine_zeus)

In [ ]:
print(df_prueba.columns.tolist())
print(df_base.columns.tolist())

['NUMERO_DOCUMENTO', 'NRG', 'FECHA_ENVIO', 'Dni', 'TIPO_GESTION', 'GESTION', 'SUBGESTION', 'SERVICIO', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'RETIRO', 'TIPO', 'LINEA_EfectivaN', 'RECORRIDO', 'CET', 'CONT_GEN', 'REGION', 'RANGO_OFERTA', 'RANGO_TASA', 'EDAD', 'Hora_Llamada', 'Prioridad', 'MARCA', 'MARCA2', 'PERFIL', 'Mejor_Telefono', 'TELEFONO', 'SOLO FH1', 'SOLO FH2', 'SOLO FH3', 'DOBLE FH', 'TRIPLE FH', 'FECHA_LLAMADA', 'DIA', 'Nombre_Campana', 'DESCRIPCION', 'DESCRIPCION2', 'Ejecutivo', 'NUMXDNI', 'SUPERVISOR', 'FLG_REP_MES_ANT', 'FLG_REP_MESES_ANT', 'tMesGestion', 'FLG_SIN_ENR', 'PROVINCIA', 'segundos', 'VAL', 'Numero_Campana', 'AGENDADOS', 'Estado', 'CntEstado', 'MntOferta', 'CNTVTAS', 'NUM_DIA_HABIL', 'Semana_Mes', 'llave', 'tMontoDesem', 'CNT_LLAMADAS', 'AREAEFECTINEGOCIOS', 'REP1', 'REP2', 'REP3', 'CONJUNTO3MESES']


In [64]:
cols_prueba = set(df_prueba.columns)
cols_base = set(df_base.columns)

solo_en_base = cols_prueba - cols_base-set(['RANGO_TASA'])
print("Solo falta estas columnas en base:", solo_en_base)

solo_en_prueba_maeba =  cols_base-cols_prueba
print("Solo falta estas columnas en maeba gestiones:", solo_en_prueba_maeba)

Solo falta estas columnas en base: {'CNT_LLAMADAS', 'SOLO FH1', 'VAL', 'llave', 'SOLO FH3', 'NRG', 'RANGO_OFERTA', 'CntEstado', 'TRIPLE FH', 'NUM_DIA_HABIL', 'SOLO FH2', 'EDAD', 'LINEA_EfectivaN', 'Semana_Mes', 'DOBLE FH', 'Numero_Campana'}
Solo falta estas columnas en maeba gestiones: {'ULTIMOMONTODESEMBOLSADOPLUS20', 'IMPDHM', 'VENTA_FUGAS', 'VENTA', 'DEVUELTO', 'RANGO_IMPDHM', 'ULTIMOMONTODESEMBOLSADO', 'TIPO_BASE', 'prioridad', 'TIPOBASEMICROPEQUENA', 'DNI_DES', 'DISTRITO', 'PERFILINICIAL', 'flag_recurrencia_efectinegocio', 'DNI_VEN', 'DNI_VICI', 'DEPARTAMENTO', 'recompra', 'Zona', 'num_cuotas_pagadas', 'flag_consentimiento', 'VENTA_DESM', 'ZONA_PROV'}


In [45]:
# Convertir las columnas a conjuntos
cols_prueba = set(df_prueba.columns)
cols_base = set(df_base.columns)

# Columnas que están en df_prueba pero no en df_base
solo_prueba = sorted(cols_prueba - cols_base)

# Columnas que están en df_base pero no en df_prueba
solo_base = sorted(cols_base - cols_prueba)

# Columnas que están en ambos DataFrames
comunes = sorted(cols_prueba & cols_base)

print(f"Columnas solo en df_prueba ({len(solo_prueba)}):")
print(solo_prueba)

print(f"\nColumnas solo en df_base ({len(solo_base)}):")
print(solo_base)

print(f"\nColumnas en común ({len(comunes)}):")
print(comunes)

Columnas solo en df_prueba (22):
['CNTVTAS', 'CNT_LLAMADAS', 'CONT_GEN', 'CntEstado', 'DOBLE FH', 'EDAD', 'LINEA_EfectivaN', 'MARCA', 'MntOferta', 'NRG', 'NUM_DIA_HABIL', 'Numero_Campana', 'RANGO_OFERTA', 'RANGO_TASA', 'SOLO FH1', 'SOLO FH2', 'SOLO FH3', 'Semana_Mes', 'TRIPLE FH', 'VAL', 'llave', 'tMontoDesem']

Columnas solo en df_base (25):
['DEPARTAMENTO', 'DEVUELTO', 'DISTRITO', 'DNI_DES', 'DNI_VEN', 'DNI_VICI', 'IMPDHM', 'MONTO_NETO', 'PERFILINICIAL', 'Proceso', 'RANGO_IMPDHM', 'TIPOBASEMICROPEQUENA', 'TIPO_BASE', 'ULTIMOMONTODESEMBOLSADO', 'ULTIMOMONTODESEMBOLSADOPLUS20', 'VENTA', 'VENTA_DESM', 'VENTA_FUGAS', 'ZONA_PROV', 'Zona', 'flag_consentimiento', 'flag_recurrencia_efectinegocio', 'num_cuotas_pagadas', 'prioridad', 'recompra']

Columnas en común (41):
['AGENDADOS', 'AREAEFECTINEGOCIOS', 'AÑO_DURACION_BASE', 'CET', 'CONJUNTO3MESES', 'DESCRIPCION', 'DESCRIPCION2', 'DIA', 'Dni', 'Ejecutivo', 'Estado', 'FECHA_ENVIO', 'FECHA_LLAMADA', 'FLG_REP_MESES_ANT', 'FLG_REP_MES_ANT', 'FLG_

In [ ]:
from sqlalchemy import text

with engine_zeus.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS ODIN.dbo.tNumEfeNegocios"))
    conn.execute(text("DROP TABLE IF EXISTS ODIN.dbo.temp_llamadas"))

query = f"""
    select NUMDOCUMENTO as DNI,count(*) as NUMXDNI 
    from ODIN.dbo.tNumeroEfectivaNegocios 
    group by NUMDOCUMENTO
"""
df_num_dni= pd.read_sql(query, engine_zeus)

if not df_num_dni.empty:
    df_num_dni.to_sql(
        name="NumxDnixEfectivaN",
        con=engine_zeus,
        if_exists="replace",
        index=False,
        chunksize=1000
    )
    print('ok')

query = f"""
    SELECT DNI AS DNI_LLAMADAS, COUNT(*) AS CNT_LLAMADAS1 
    FROM maeba.ADM_OBJ_TG.synTmpLlamadasGeneralEfectivaN
    GROUP BY DNI
"""
df_num_dni= pd.read_sql(query, engine_zeus)

if not df_num_dni.empty:
    df_num_dni.to_sql(
        name="temp_llamadas",
        con=engine_zeus,
        if_exists="replace",
        index=False,
        chunksize=1000
    )
    print('ok')


NameError: name 'engine_zeus' is not defined

In [ ]:
from sqlalchemy import text
import pandas as pd

with engine_zeus.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS ODIN.dbo.NumxDnixEfectivaN"))
    conn.execute(text("DROP TABLE IF EXISTS ODIN.dbo.temp_llamadas"))

query_num_dni = """
    SELECT 
        NUMDOCUMENTO AS DNI,
        COUNT(*) AS NUMXDNI 
    FROM ODIN.dbo.tNumeroEfectivaNegocios 
    GROUP BY NUMDOCUMENTO
"""

df_num_dni = pd.read_sql(query_num_dni, engine_zeus)

if not df_num_dni.empty:
    df_num_dni.to_sql(
        name="NumxDnixEfectivaN",
        con=engine_zeus,
        schema="dbo",
        if_exists="replace",
        index=False,
        chunksize=1000
    )
    print("NumxDnixEfectivaN ok:", len(df_num_dni))

query_llamadas = """
    SELECT 
        DNI,
        COUNT(*) AS CNT_LLAMADAS
    FROM maeba.ADM_OBJ_TG.synTmpLlamadasGeneralEfectivaN
    GROUP BY DNI
"""

df_llamadas = pd.read_sql(query_llamadas, engine_zeus)

if not df_llamadas.empty:
    df_llamadas.to_sql(
        name="temp_llamadas",
        con=engine_zeus,
        schema="dbo",
        if_exists="replace",
        index=False,
        chunksize=1000
    )
    print("temp_llamadas ok:", len(df_llamadas))

NameError: name 'engine_zeus' is not defined

In [72]:
df_llamadas.head()

,DNI,CNT_LLAMADAS1
0,20888409,13
1,44628750,16
2,44872888,37
3,40693042,7
4,48435403,18


In [74]:
query = f"""
	SELECT 
		A.NUMDOCUMENTO as NUMERO_DOCUMENTO,
		CASE WHEN A.NUMDOCUMENTO IS NULL THEN B.Dni ELSE NULL END AS DNI_VICI, 
		CASE WHEN A.NUMDOCUMENTO IS NULL THEN ven.Dni ELSE NULL END AS DNI_VEN, 
		CASE WHEN A.NUMDOCUMENTO IS NULL THEN desem.Dni ELSE NULL END AS DNI_DES, 
		CASE
			WHEN A.NUMDOCUMENTO IS NULL THEN 'FUERA DE BASE'
			ELSE 'EN BASE'
		END as TIPO_BASE,
		a.SEGMICROPEQUENACOMERCIAL as Prioridad,
		a.TIPOBASEMICROPEQUENA,
		a.DEPARTAMENTO,
		a.PROVINCIA,
		a.DISTRITO,
		a.AREAEFECTINEGOCIOS,
		a.num_cuotas_pagadas,
		a.prioridad,
		a.PERFILINICIAL,
		a.ULTIMOMONTODESEMBOLSADO,
		a.ULTIMOMONTODESEMBOLSADOPLUS20,
		a.flag_recurrencia_efectinegocio,
		a.RETIRO,
		a.DEVUELTO,
		a.recompra,
		a.Proceso as MARCA, --MARCA
		a.Zona,
		a.flag_consentimiento,
		A.SERVICIO,
		A.AÑO_DURACION_BASE,
		A.MES_DURACION_BASE,
		A.RETIRO,
		cast(a.IMPDHM as numeric(18,2)) as IMPDHM,
		CASE
			WHEN cast(a.IMPDHM as numeric(18,2)) IS NULL THEN '14.SIN DATO'
			WHEN cast(a.IMPDHM as numeric(18,2)) < 10000 THEN '00.[0 - 10,000>'
			WHEN cast(a.IMPDHM as numeric(18,2)) < 20000 THEN '01.[10,000 - 20,000>'
			WHEN cast(a.IMPDHM as numeric(18,2)) < 30000 THEN '02.[20,000 - 30,000>'
			WHEN cast(a.IMPDHM as numeric(18,2)) < 40000 THEN '03.[30,000 - 40,000>'
			WHEN cast(a.IMPDHM as numeric(18,2)) < 50000 THEN '04.[40,000 - 50,000>'
			WHEN cast(a.IMPDHM as numeric(18,2)) < 60000 THEN '05.[50,000 - 60,000>'
			WHEN cast(a.IMPDHM as numeric(18,2)) < 70000 THEN '06.[60,000 - 70,000>'
			WHEN cast(a.IMPDHM as numeric(18,2)) < 80000 THEN '07.[70,000 - 80,000>'
			WHEN cast(a.IMPDHM as numeric(18,2)) < 90000 THEN '08.[80,000 - 90,000>'
			WHEN cast(a.IMPDHM as numeric(18,2)) < 100000 THEN '09.[90,000 - 100,000>'
			WHEN cast(a.IMPDHM as numeric(18,2)) < 150000 THEN '10.[100,000 - 150,000>'
			WHEN cast(a.IMPDHM as numeric(18,2)) < 200000 THEN '11.[150,000 - 200,000>'
			WHEN cast(a.IMPDHM as numeric(18,2)) < 250000 THEN '12.[200,000 - 250,000>'
			WHEN cast(a.IMPDHM as numeric(18,2)) >= 250000 THEN '13.[250,000 A MÁS>'
		END AS RANGO_IMPDHM,
		CASE 
			WHEN a.DEPARTAMENTO IN ('LIMA','CALLAO') THEN 'LIMA' 
			WHEN a.DEPARTAMENTO is null THEN 'SIN DATOS' 
			ELSE 'PROVINCIA' 
		END AS ZONA_PROV,
		CASE	
			WHEN ISNULL(A.[PROVINCIA],'') ='' THEN 'Leads sin región'
			WHEN A.[PROVINCIA] ='LIMA' OR A.[PROVINCIA] ='CALLAO' THEN 'Leads Lima'
			WHEN A.[PROVINCIA] <>'LIMA' THEN 'Leads Provincia' 
		end as REGION,
		CONVERT( VARCHAR(12),A.FECHA_ENVIO,103) AS FECHA_ENVIO,
		CASE WHEN (A.REP1=1 OR A.REP2=1 /*OR A.REP3=1*/) THEN 'Stock' ELSE 'Nuevo' END AS CONJUNTO3MESES,
		case when isnull(B.Estado_,'') ='' then 'NO DISCADO' ELSE B.Estado_ END AS TIPO_GESTION,
		case when isnull(B.Sub_Estado_,'') ='' then 'NO DISCADO' ELSE B.Sub_Estado_ END AS GESTION,
		Case when isnull(B.Descripcion_,'') ='' then 'NO DISCADO' ELSE B.Descripcion_ END AS SUBGESTION,
		B.Dni,
		CASE 
			WHEN ISNULL(FLAT2,0) =0 THEN 'NO ENRIQUECIDO'
			WHEN ISNULL(B.Codigo_Paleta,'')='' THEN  'NO DISCADO' 
			ELSE 'GEST+DISC' END as TIPO,
	RECORRIDO=CASE  WHEN B.DNI IS NULL THEN 0 ELSE 1 END,
	CASE 
		WHEN ven.dni IS not NULL THEN 1
		WHEN B.Estado_ in ('CONTACTO EFECTIVO CON TITULAR','CONTACTO EFECTIVO') THEN 1 
		ELSE 0
	END as CET,	
	CASE 
		WHEN ven.dni IS not NULL THEN 1
		WHEN B.Estado_ in ('CONTACTO EFECTIVO CON TITULAR','CONTACTO EFECTIVO') 
			and not Descripcion_ in ('CLIENTE DESEA QUE NO VUELVAN A LLAMAR','NO BRINDA CONSENTIMIENTO','CLIENTE CORTO SIN ESCUCHAR OFERTA')
		THEN 1 ELSE 0 
	END as CONT_GEN,--Estado
	case when ven.dni is null then 0 else 1 end as CNTVTAS ,
	isnull(replace(ven.ESTADO,'-','EN PROCESO'),'EN PROCESO') as Estado,
	b.Hora_Llamada,
	CAST(NULL AS VARCHAR(200)) as MARCA2,
	cast(null as varchar(200)) AS PERFIL,
	b.Mejor_Telefono,
	b.PHONE_NUMBER as TELEFONO,
	b.Fecha_Llamada AS FECHA_LLAMADA,
	substring(convert(varchar,b.Fecha_Llamada),9,2) as DIA,
	cast(null as varchar(200)) as Nombre_Campana,
	'REGISTROS ENTREGADOS'  AS 'DESCRIPCION',
	'REGISTROS CARGADOS'  AS 'DESCRIPCION2',
	isnull(isnull(d.tNombreCompleto,b.Ejecutivo),ven.Ejecutivo) as Ejecutivo,
	c.NUMXDNI,
	ISNULL(d.tSupervisor,'Sin_Asignar') as SUPERVISOR,
	FLG_REP_MES_ANT=CASE WHEN ISNULL(REP1,0)=1 THEN 'Stock' Else 'Nuevo' END,
	FLG_REP_MESES_ANT=CASE WHEN ISNULL(REP2,0)=1 THEN 'Stock' Else 'Nuevo' END, 
	DATENAME(MONTH,GETDATE()) AS tMesGestion,
	FLG_SIN_ENR =CASE WHEN ISNULL(FLAT2,0) =0 THEN 'SIN ENRIQUECER' END,
	a.PROVINCIA,b.segundos, 
	case 
	--	WHEN ven.dni IS not NULL THEN 0
		when B.Descripcion_='VOLVER A LLAMAR' then 1 
		else 0 
	end as AGENDADOS,
	CASE WHEN A.REP1=1 THEN 'Stock' ELSE 'NUEVO' END AS REP1,
	CASE WHEN A.REP2=1 THEN 'Stock' ELSE 'NUEVO' END AS REP2,
	/*CASE WHEN A.REP3=1 THEN 'Stock' ELSE 'NUEVO' END */
	'' AS REP3,
	CASE WHEN ven.dni IS NULL THEN 0 ELSE 1 END AS VENTA,
	convert(int,replace(replace(replace(REPLACE(REPLACE(ven.MONTO,'.',''),',',''),'	',''),'S/. ',''),'S/ ','')) as MntOferta,
	CASE WHEN desem.DNI IS NOT NULL AND desem.Resolucion='TARGET' THEN 1 ELSE 0 END AS VENTA_DESM,
	CASE WHEN desem.Resolucion!='TARGET' THEN 1 ELSE 0 END AS VENTA_FUGAS,
	desem.MONTO_NETO as tMontoDesem,
	llam.CNT_LLAMADAS1
	FROM [ODIN].[dbo].[Base_Maestra_Efectiva_Negocios_Vigente] a 
	left join [THOTH].[dbo].[Tmp_LLamadas_Efectiva_Negocios_5] b 
		on a.NUMDOCUMENTO=b.Dni
		and b.Fecha_Llamada>='{fecha_desembolso}'
		and b.Fecha_Llamada<=EOMONTH('{fecha_desembolso}')
	left join ODIN.dbo.temp_llamadas llam 
		ON a.NUMDOCUMENTO=llam.DNI		
	left join ODIN.dbo.tNumeroEfectivaNegocios C 
		ON a.NUMDOCUMENTO=c.DNI
	left join [URANO].[dbo].[tPersonal] d 
		ON b.DNI_EJECUTIVO=d.tDocumentoVici COLLATE Modern_Spanish_CI_AS
	left join SAMANTHA.dbo.efectiva_negocios_ventas ven 
		ON A.NUMDOCUMENTO=ven.Dni COLLATE Modern_Spanish_CI_AS
		and ven.FECHA>='{fecha_desembolso}'
		and ven.FECHA<=EOMONTH('{fecha_desembolso}')
	left join SAMANTHA.dbo.efectiva_negocios_ventas_desembolso desem
		ON A.NUMDOCUMENTO=desem.Dni COLLATE Modern_Spanish_CI_AS	
		and desem.FECHA_DESEMBOLSO>='{fecha_desembolso}'
		and desem.FECHA_DESEMBOLSO<=EOMONTH('{fecha_desembolso}')
	where a.NUMDOCUMENTO is not null
    """
df_base= pd.read_sql(query, engine_zeus)


ProgrammingError: (pyodbc.ProgrammingError) ('42S22', "[42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]El nombre de columna 'DNI' no es válido. (207) (SQLExecDirectW); [42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]El nombre de columna 'NUMXDNI' no es válido. (207); [42S22] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]El nombre de columna 'CNT_LLAMADAS1' no es válido. (207)")
[SQL: 
	SELECT 
		A.NUMDOCUMENTO as NUMERO_DOCUMENTO,
		CASE WHEN A.NUMDOCUMENTO IS NULL THEN B.Dni ELSE NULL END AS DNI_VICI, 
		CASE WHEN A.NUMDOCUMENTO IS NULL THEN ven.Dni ELSE NULL END AS DNI_VEN, 
		CASE WHEN A.NUMDOCUMENTO IS NULL THEN desem.Dni ELSE NULL END AS DNI_DES, 
		CASE
			WHEN A.NUMDOCUMENTO IS NULL THEN 'FUERA DE BASE'
			ELSE 'EN BASE'
		END as TIPO_BASE,
		a.SEGMICROPEQUENACOMERCIAL as Prioridad,
		a.TIPOBASEMICROPEQUENA,
		a.DEPARTAMENTO,
		a.PROVINCIA,
		a.DISTRITO,
		a.AREAEFECTINEGOCIOS,
		a.num_cuotas_pagadas,
		a.prioridad,
		a.PERFILINICIAL,
		a.ULTI

In [40]:
print(df_base.columns.to_list())

['NUMERO_DOCUMENTO', 'DNI_VICI', 'DNI_VEN', 'DNI_DES', 'TIPO_BASE', 'Prioridad', 'TIPOBASEMICROPEQUENA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'AREAEFECTINEGOCIOS', 'num_cuotas_pagadas', 'prioridad', 'PERFILINICIAL', 'ULTIMOMONTODESEMBOLSADO', 'ULTIMOMONTODESEMBOLSADOPLUS20', 'flag_recurrencia_efectinegocio', 'RETIRO', 'DEVUELTO', 'recompra', 'Proceso', 'Zona', 'flag_consentimiento', 'SERVICIO', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'RETIRO', 'IMPDHM', 'RANGO_IMPDHM', 'ZONA_PROV', 'REGION', 'FECHA_ENVIO', 'CONJUNTO3MESES', 'TIPO_GESTION', 'GESTION', 'SUBGESTION', 'Dni', 'TIPO', 'RECORRIDO', 'CET', 'Estado', 'Hora_Llamada', 'MARCA2', 'PERFIL', 'Mejor_Telefono', 'TELEFONO', 'FECHA_LLAMADA', 'DIA', 'Nombre_Campana', 'DESCRIPCION', 'DESCRIPCION2', 'Ejecutivo', 'NUMXDNI', 'SUPERVISOR', 'FLG_REP_MES_ANT', 'FLG_REP_MESES_ANT', 'tMesGestion', 'FLG_SIN_ENR', 'PROVINCIA', 'segundos', 'AGENDADOS', 'REP1', 'REP2', 'REP3', 'VENTA', 'VENTA_DESM', 'VENTA_FUGAS', 'MONTO_NETO']


In [ ]:
['NUMERO_DOCUMENTO', 'DNI_VICI', 'DNI_VEN', 'DNI_DES', 'TIPO_BASE', 'Prioridad', 'TIPOBASEMICROPEQUENA', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'AREAEFECTINEGOCIOS', 'num_cuotas_pagadas', 'prioridad', 'PERFILINICIAL', 'ULTIMOMONTODESEMBOLSADO', 'ULTIMOMONTODESEMBOLSADOPLUS20', 'flag_recurrencia_efectinegocio', 'RETIRO', 'DEVUELTO', 'recompra', 'Proceso', 'Zona', 'flag_consentimiento', 'SERVICIO', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'RETIRO', 'IMPDHM', 'RANGO_IMPDHM', 'ZONA_PROV', 'REGION', 'FECHA_ENVIO', 'CONJUNTO3MESES', 'TIPO_GESTION', 'GESTION', 'SUBGESTION', 'Dni', 'TIPO', 'RECORRIDO', 'CET', 'Estado', 'Hora_Llamada', 'MARCA2', 'PERFIL', 'Mejor_Telefono', 'TELEFONO', 'FECHA_LLAMADA', 'DIA', 'Nombre_Campana', 'DESCRIPCION', 'DESCRIPCION2', 'Ejecutivo', 'NUMXDNI', 'SUPERVISOR', 'FLG_REP_MES_ANT', 'FLG_REP_MESES_ANT', 'tMesGestion', 'FLG_SIN_ENR', 'PROVINCIA', 'segundos', 'AGENDADOS', 'REP1', 'REP2', 'REP3', 'VENTA', 'VENTA_DESM', 'VENTA_FUGAS', 'MONTO_NETO']

In [18]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

df_desembolso_cli.head(2)

,DNI,PRESTAMO,FECHA_DESEMBOLSO,PLAZO,MONTO,SEGURO,MONTO_NETO,PRODUCTO,NOM_PRODUCTO,TASA_NOMINAL,EMPRESA,AGENCIA,COD_AGENCIA,DEPARTAMENTO,PROVINCIA,DISTRITO,PERFIL_DETALLE,COD_VENDEDOR,NOM_VENDEDOR,USUARIO_FDN,NOM_FDN,PERFIL,Plaza_Creditos,ZONA_CREDITOS,cantcruces,Resolucion,AREAEFECTINEGOCIOS,CODUSUARIOFDN,FUNCIONARIO,CAMPAÑA
0,44386328,2601757,2026-06-23,12,20001.0,NaN,20001.0,3031,SIN CRECER CT,20.879999,EFE,JULIACA,110,NaN,NaN,NaN,PREFERENTE,NaN,NaN,lsonco,NaN,NUEVO,JULIACA,SUR,2,Target,JULIACA,lsonco,NaN,junio 2026
1,40347881,2602048,2026-06-23,12,30000.0,NaN,30000.0,3088,CRECER CT CREDITO,35.582401,EFE,SAN JUAN LURIGANCHO,551,NaN,NaN,NaN,PREFERENTE,NaN,NaN,cmezab,NaN,NUEVO,SAN JUAN DE LURIGANCHO,LIMA NORESTE,2,Target,JUNIN,cmezab,NaN,junio 2026


In [ ]:

query = """
    SELECT * FROM valentina.dbo.efectiva_negocios_ventas_desembolso
    WHERE FECHA_DESEMBOLSO >= '2026-06-01'
    AND FECHA_DESEMBOLSO < '2026-07-01'
  """
df_desembolso_target= pd.read_sql(query,engine_sa)

query = """
  SELECT 
      FECHA,
      dni,
      CLIENTE,
      PROMOTOR,
      ESTADO,
      AUDITOR,
      MES_VENTA,
      CAMPANA,
      DIA,
      TRAMA_HORA,
      CANDADO,
      MONTO,
      EJECUTIVO,
      TRAMA_HORA_Val,
      ANIO,
      DNIEjecutivo,
      celular
  FROM SAMANTHA.dbo.Ventas_Target A
      WHERE CAMPANA = 'negocios'
        AND FECHA >= '2026-06-01'
  """
df_venta_target= pd.read_sql(query,engine_zeus)


In [45]:
print(df_venta_cli.shape)
print(df_gestiones_dia_cli.shape)
print(df_gestiones_app_cli.shape)
print(df_desembolso_target.shape)
print(df_venta_target.shape)

(113, 29)
(437, 3)
(5170, 15)
(9, 30)
(3511, 19)


In [ ]:
from sqlalchemy import text

with engine_samantha.begin() as conn:
    conn.execute(text("delete from SAMANTHA.dbo.efectiva_negocios_ventas where tipo_venta_ref is not null"))

query = """
  SELECT 
      A.FECHA,
      A.dni,
      A.CLIENTE,
      A.PROMOTOR,
      A.ESTADO,
      A.AUDITOR,
      A.MES_VENTA,
      A.CAMPANA,
      A.DIA,
      A.TRAMA_HORA,
      A.CANDADO,
      A.MONTO,
      A.EJECUTIVO,
      A.TRAMA_HORA_Val,
      A.ANIO,
      A.DNIEjecutivo,
      A.celular,
      'target' as tipo_venta_ref,
      R.fecha_ref
  FROM SAMANTHA.dbo.Ventas_Target A
  CROSS JOIN (
      SELECT MAX(FECHA) AS fecha_ref
      FROM SAMANTHA.dbo.efectiva_negocios_ventas
      WHERE CAMPANA = 'negocios'
        AND FECHA >= '2026-06-01'
  ) R
  WHERE A.CAMPANA = 'negocios'
    AND A.FECHA >= '2026-06-01'
    AND NOT EXISTS (
          SELECT 1
          FROM SAMANTHA.dbo.efectiva_negocios_ventas B
          WHERE B.DNI = A.DNI COLLATE Modern_Spanish_CI_AS
            AND B.FECHA >= '2026-06-01'
    )
  """
df_venta_target= pd.read_sql(query,engine_zeus)

df_venta_target["FECHA"] = pd.to_datetime(df_venta_target["FECHA"])
df_venta_target = (
    df_venta_target
    .sort_values("FECHA", ascending=False)
    .drop_duplicates(subset="dni", keep="first")
    .reset_index(drop=True)
)

df_venta_target = df_venta_target[
    (df_venta_target["fecha_ref"].isna()) |
    (df_venta_target["FECHA"] > df_venta_target["fecha_ref"])
].reset_index(drop=True)
df_venta_target=df_venta_target.drop(columns='fecha_ref')

if not df_venta_target.empty:
    df_venta_target.to_sql(
        name="efectiva_negocios_ventas",
        con=engine_samantha,
        if_exists="append",
        index=False,
        chunksize=1000
    )
    print('ok')



In [ ]:
from sqlalchemy import text

with engine_zeus.begin() as conn:
    conn.execute(text("truncate table ODIN.dbo.tNumeroEfectivaNegocios"))

query = f"""
    select NUMDOCUMENTO as DNI,count(*) as NUMXDNI 
    from ODIN.dbo.tNumeroEfectivaNegocios 
    group by NUMDOCUMENTO
"""
df_num_dni= pd.read_sql(query, engine_zeus)

if not df_num_dni.empty:
    df_num_dni.to_sql(
        name="NumxDnixEfectivaN",
        con=engine_zeus,
        if_exists="replace",
        index=False,
        chunksize=1000
    )
    print('ok')



In [4]:
dni_duplicados = (
    df_base['NUMERO_DOCUMENTO']
    .value_counts()
    .loc[lambda x: x > 1]
)
dni_duplicados.head()

Series([], Name: count, dtype: int64)

In [5]:
print(
    df_base.loc[
        df_base["NUMERO_DOCUMENTO"].isna(),
        ["NUMERO_DOCUMENTO", "DNI_VEN", "DNI_DES", "DNI_VICI"]
    ].drop_duplicates()
)

       NUMERO_DOCUMENTO DNI_VEN   DNI_DES  DNI_VICI
146800             None    None      None  10708172
146801             None    None      None  41897609
146802             None    None      None  20718426
146803             None    None      None  70141169
146804             None    None      None  44699146
...                 ...     ...       ...       ...
183327             None    None  42538058      None
183328             None    None  09683837      None
183329             None    None  24813547      None
183330             None    None  44787949      None
183331             None    None  41281436      None

[22417 rows x 4 columns]


In [16]:
df_base[['NUMERO_DOCUMENTO','DNI']].head()

KeyError: "['DNI'] not in index"

In [7]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

df_base[(df_base['NUMERO_DOCUMENTO']=='10708172')|(df_base['DNI_VICI']=='10708172')].head(10)

,NUMERO_DOCUMENTO,DNI_VICI,DNI_VEN,DNI_DES,TIPO_BASE,Prioridad,TIPOBASEMICROPEQUENA,DEPARTAMENTO,PROVINCIA,DISTRITO,AREAEFECTINEGOCIOS,num_cuotas_pagadas,prioridad,PERFILINICIAL,ULTIMOMONTODESEMBOLSADO,ULTIMOMONTODESEMBOLSADOPLUS20,flag_recurrencia_efectinegocio,RETIRO,DEVUELTO,recompra,Proceso,Zona,flag_consentimiento,SERVICIO,AÑO_DURACION_BASE,MES_DURACION_BASE,RETIRO,IMPDHM,RANGO_IMPDHM,ZONA_PROV,REGION,FECHA_ENVIO,CONJUNTO3MESES,TIPO_GESTION,GESTION,SUBGESTION,Dni,TIPO,RECORRIDO,CET,Estado,Hora_Llamada,MARCA2,PERFIL,Mejor_Telefono,TELEFONO,FECHA_LLAMADA,DIA,Nombre_Campana,DESCRIPCION,DESCRIPCION2,Ejecutivo,NUMXDNI,SUPERVISOR,FLG_REP_MES_ANT,FLG_REP_MESES_ANT,tMesGestion,FLG_SIN_ENR,PROVINCIA,segundos,AGENDADOS,REP1,REP2,REP3,VENTA,VENTA_DESM,VENTA_FUGAS,MONTO_NETO
146800,None,10708172,None,None,FUERA DE BASE,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,NaN,14.SIN DATO,SIN DATOS,Leads sin región,None,Nuevo,NO CONTACTO,SIN CONTACTO,TONO OCUPADO,10708172,NO ENRIQUECIDO,1,0,0,10.0,None,None,1.0,989866672,2026-06-05,05,None,REGISTROS ENTREGADOS,REGISTROS CARGADOS,Outbound Auto Dial,NaN,Sin_Asignar,Nuevo,Nuevo,Junio,SIN ENRIQUECER,None,0.0,0,NUEVO,NUEVO,,0,0,0,NaN


In [ ]:
SELECT 
1 AS NRG,
A.NUMDOCUMENTO as NUMERO_DOCUMENTO,
a.SEGMICROPEQUENACOMERCIAL as Prioridad,
a.IMPDHM,
    CASE
        WHEN a.IMPDHM IS NULL THEN 'SIN DATO'
        WHEN a.IMPDHM < 10000 THEN '00.<0 - 10,000]'
        WHEN a.IMPDHM < 20000 THEN '01.<10,000 - 20,000]'
        WHEN a.IMPDHM < 30000 THEN '02.<20,000 - 30,000]'
        WHEN a.IMPDHM < 40000 THEN '03.<30,000 - 40,000]'
        WHEN a.IMPDHM < 50000 THEN '04.<40,000 - 50,000]'
        WHEN a.IMPDHM < 60000 THEN '05.<50,000 - 60,000]'
        WHEN a.IMPDHM < 70000 THEN '06.<60,000 - 70,000]'
        WHEN a.IMPDHM < 80000 THEN '07.<70,000 - 80,000]'
        WHEN a.IMPDHM < 90000 THEN '08.<80,000 - 90,000]'
        WHEN a.IMPDHM < 100000 THEN '09.<90,000 - 100,000]'
        WHEN a.IMPDHM < 150000 THEN '10.<100,000 - 150,000]'
        WHEN a.IMPDHM < 200000 THEN '11.<150,000 - 200,000]'
        WHEN a.IMPDHM < 250000 THEN '12.<200,000 - 250,000]'
        WHEN a.IMPDHM > 250000 THEN '13.<250,000 A MÁS]'
    END AS RANGO_IMPDHM,
	a.TIPOBASEMICROPEQUENA,
    a.DEPARTAMENTO,
	CASE 
		WHEN a.DEPARTAMENTO IN ('LIMA','CALLAO') THEN 'LIMA' 
		WHEN a.DEPARTAMENTO is null THEN 'SIN DATOS' 
		ELSE 'PROVINCIA' 
	END AS ZONA_PROV,
	CASE	
		WHEN ISNULL(A.[PROVINCIA],'') ='' THEN 'Leads sin región'
		WHEN A.[PROVINCIA] ='LIMA' OR A.[PROVINCIA] ='CALLAO' THEN 'Leads Lima'
		WHEN A.[PROVINCIA] <>'LIMA' THEN 'Leads Provincia' 
	end as REGION,
    a.PROVINCIA,
    a.DISTRITO,
	CONVERT( VARCHAR(12),A.FECHA_ENVIO,103) AS FECHA_ENVIO,
    a.AREAEFECTINEGOCIOS,
    a.num_cuotas_pagadas,
    a.num_cuotas_impagas,
    a.prioridad,
    a.PERFILINICIAL,
    a.ULTIMOMONTODESEMBOLSADO,
    a.ULTIMOMONTODESEMBOLSADOPLUS20,
    a.flag_recurrencia_efectinegocio,
    a.RETIRO,
    a.DEVUELTO,
    a.recompra,
    a.Proceso, --MARCA
    a.Zona,
    a.flag_consentimiento,
	A.SERVICIO,
	A.AÑO_DURACION_BASE,
	A.MES_DURACION_BASE,
	A.RETIRO,
CASE WHEN (A.REP1=1 OR A.REP2=1 /*OR A.REP3=1*/) THEN 'Stock' ELSE 'Nuevo' END AS CONJUNTO3MESES,
case when isnull(B.Estado_,'') ='' then 'NO DISCADO' ELSE B.Estado_ END AS TIPO_GESTION,
case when isnull(B.Sub_Estado_,'') ='' then 'NO DISCADO' ELSE B.Sub_Estado_ END AS GESTION,
Case when isnull(B.Descripcion_,'') ='' then 'NO DISCADO' ELSE B.Descripcion_ END AS SUBGESTION,
B.Dni,
'TIPO'=CASE 
		WHEN ISNULL(FLAT2,0) =0 THEN 'NO ENRIQUECIDO'
		WHEN ISNULL(B.Codigo_Paleta,'')='' THEN  'NO DISCADO' 
		ELSE 'GEST+DISC' END,
RECORRIDO=CASE  WHEN B.DNI IS NULL THEN 0 ELSE 1 END,
CASE WHEN B.Estado_ in ('CONTACTO EFECTIVO CON TITULAR','CONTACTO EFECTIVO') THEN 1 ELSE 0 END as CET,	
CASE 
	WHEN B.Estado_ in ('CONTACTO EFECTIVO CON TITULAR','CONTACTO EFECTIVO') 
				and not Descripcion_ in ('CLIENTE DESEA QUE NO VUELVAN A LLAMAR','NO BRINDA CONSENTIMIENTO','CLIENTE CORTO SIN ESCUCHAR OFERTA')
	THEN 1 ELSE 0 
END as CONT_GEN,
b.Hora_Llamada,
CAST(NULL AS VARCHAR(200)) as MARCA2,
cast(null as varchar(200)) AS PERFIL,
b.Mejor_Telefono,
b.PHONE_NUMBER as TELEFONO,
b.Fecha_Llamada AS FECHA_LLAMADA,
substring(convert(varchar,b.Fecha_Llamada),9,2) as DIA,
cast(null as varchar(200)) as Nombre_Campana,
'REGISTROS ENTREGADOS'  AS 'DESCRIPCION',
'REGISTROS CARGADOS'  AS 'DESCRIPCION2',
isnull(d.tNombreCompleto,b.Ejecutivo) as Ejecutivo,
c.NUMXDNI,
ISNULL(d.tSupervisor,'Sin_Asignar') as SUPERVISOR,
FLG_REP_MES_ANT=CASE WHEN ISNULL(REP1,0)=1 THEN 'Stock' Else 'Nuevo' END,
FLG_REP_MESES_ANT=CASE WHEN ISNULL(REP2,0)=1 THEN 'Stock' Else 'Nuevo' END, 
DATENAME(MONTH,GETDATE()) AS tMesGestion,
FLG_SIN_ENR =CASE WHEN ISNULL(FLAT2,0) =0 THEN 'SIN ENRIQUECER' END,
a.PROVINCIA,b.segundos, 
case 
	when 
		B.Estado_ in ('CONTACTO EFECTIVO CON TITULAR','CONTACTO EFECTIVO') 
		and not Descripcion_ in ('CLIENTE DESEA QUE NO VUELVAN A LLAMAR','NO BRINDA CONSENTIMIENTO','CLIENTE CORTO SIN ESCUCHAR OFERTA') 
	then 1 
	else 0 
end AS VAL,
case when B.Descripcion_='VOLVER A LLAMAR' then 1 else 0 end as AGENDADOS,
cast(null as varchar(100)) Estado,
cast(null as NUMERIC) CntEstado,
cast(null as NUMERIC) MntOferta,
cast(null as int) CNTVTAS,
cast(null as int)[NUM_DIA_HABIL],
cast(null as varchar(100)) [Semana_Mes],
cast(null as float) tMontoDesem,
CASE WHEN A.REP1=1 THEN 'Stock' ELSE 'NUEVO' END AS REP1,
CASE WHEN A.REP2=1 THEN 'Stock' ELSE 'NUEVO' END AS REP2,
/*CASE WHEN A.REP3=1 THEN 'Stock' ELSE 'NUEVO' END */
'' AS REP3,
CASE WHEN ven.ind_venta IS NULL THEN 0 ELSE 1 END AS VENTA,
CASE WHEN desem.DNI IS NOT NULL AND desem.Resolucion='TARGET' THEN 1 ELSE 0 END AS VENTA_DESM,
CASE WHEN desem.Resolucion!='TARGET' THEN 1 ELSE 0 END AS VENTA_FUGAS,
desem.MONTO_NETO
FROM [ODIN].[dbo].[Base_Maestra_Efectiva_Negocios_Vigente] a 
FULL OUTER JOIN [THOTH].[dbo].[Tmp_LLamadas_Efectiva_Negocios_5] b 
on a.NUMDOCUMENTO=b.Dni
FULL OUTER JOIN ODIN.dbo.NumxDnixEfectivaN C 
ON a.NUMDOCUMENTO=c.DNI
FULL OUTER JOIN [URANO].[dbo].[tPersonal] d 
ON b.DNI_EJECUTIVO=d.tDocumentoVici COLLATE Modern_Spanish_CI_AS
FULL OUTER JOIN SAMANTHA.dbo.efectiva_negocios_ventas ven 
ON A.NUMDOCUMENTO=B.Dni
FULL OUTER JOIN SAMANTHA.dbo.efectiva_negocios_ventas_desembolso desem
ON A.NUMDOCUMENTO=B.Dni



/*
SELECT *
FROM sys.synonyms
WHERE name = 'synNumeroEfectivaNegocios';

*/



In [ ]:
# def since_base_maestra_efe_negocio(spark):
query = """
    Select  
    ROW_NUMBER() OVER (ORDER BY (SELECT NULL)) AS indice,
    ISNULL([SEGMICROPEQUENACOMERCIAL],'IMPULSA') as first_name,  
    isnull([DEPARTAMENTO],'Sin_Departamento') as last_name,  
    CASE 
        WHEN Materno IS NULL THEN Nombres +' '+Paterno 
        ELSE Nombres + ' ' + Paterno +' '+ Materno 
    END AS address1,  
    [DIRECCION] as address2,  
    'EMP1='+[empresa1]+'/ EMP2='+[empresa2]+'/ EMP3='+ [empresa3] as address3,  
    Proceso as title,  
    Montos_Referenciales as comments, 
    'IMPORTE DEUDA'+' '+CONVERT(VARCHAR,IMPDHM) as security_phrase,  
    NUMDOCUMENTO as vendor_lead_code,  
    PROVINCIA as province,  
    DISTRITO as city,  
    AREAEFECTINEGOCIOS AS email,  
    IMPDHM AS deuda,  
    SEGMICROPEQUENACOMERCIAL as tip_prioridad,  
    Zona,  
    perfil_ic,  
    canal_asignado,
    CASE 
        WHEN REP1=1 then 'STOCK' 
        ELSE 'NUEVO'
    END as marca,  
    fecha_envio,      
    TIPOBASEMICROPEQUENA,     
    case
        when len(replace(retiro,' ',''))>3  then lower(replace(retiro,' ','_'))
        else 'no_aplica'
    end as retiro 
    from dantalion.dbo.Base_Maestra_Efectiva_Negocios_Vigente

select tDocumento,tNombreCompleto,tSexo,tGrupo,tCargo,tCampana,
tSupervisor,tEmpresa,tContrato,tModalidad,tCondicion,tHorario,
tCese,tEstado,tNivel1
    from [URANO].[dbo].[tPersonal]

    """
        
df_base= obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)

df_base = df_base.withColumn(
    "region",
    F.when(F.coalesce(F.col("province"), F.lit("")) == "", "Leads sin región")
    .when(F.col("province").isin("LIMA", "CALLAO"), "Leads Lima")
    .otherwise("Leads Provincia")
    )

    df_base = df_base.withColumn(
        "prioridada",
        F.when(
            (F.col("province").isin("LIMA", "CALLAO")) &
            (F.col("title") == "humano seguro") &
            (F.col("tip_prioridad") == "prf"),
            "prioridad 2"
        )
        .when(
            (F.col("province") != "LIMA") &
            (F.col("title") == "fasttrack") &
            (F.col("tip_prioridad") == "elt"),
            "prioridad 1"
        )
        .when(
            (F.col("province").isin("LIMA", "CALLAO")) &
            (F.col("title") == "FULL") &
            (F.col("tip_prioridad") == "prf"),
            "prioridad 3"
        )
        .otherwise("otra prioridad")
    )

    return df_base.withColumn(
        "vendor_lead_code",
        F.right(
            F.concat(F.lit("00000000"), F.col("vendor_lead_code")),
            F.lit(8)
        )
    )
    # print(df_base.columns)


In [ ]:

    return pd.read_sql(query, engine_sa)
    return pd.read_sql(query, engine_sa)
    return pd.read_sql(query, engine_sa)

In [ ]:

def since_target_sales(engine_zeus,fecha_mes_base,campana):
    query = f"""
        select FECHA as fecha_gestion,DNI as vendor_lead_code,
        PROMOTOR as promotor,
        ESTADO as estado_venta,
        TRAMA_HORA as tramo_venta,
        MONTO as monto_venta,
        DNIEjecutivo as dni_ejecutivo_venta,
        Producto as producto_venta,
        Producto as title,
        Celular as cel_venta,1 as venta,
        subcampana
        from SAMANTHA.dbo.Ventas_Target
        where campana='{campana}'
        and fecha >= DATEADD(MONTH, -4, DATEADD(DAY, 1, EOMONTH('{fecha_mes_base}'))) 
        and fecha < DATEADD(MONTH, 0, DATEADD(DAY, 1, EOMONTH('{fecha_mes_base}')))
        """
    return pd.read_sql(query, engine_zeus)

def since_target_disbursements(engine_sa,fecha_mes_base,campana):
    query = f"""
        select cast(FECHA_HORA_DESEMBOLSO as date) as fecha_desembolso,
        CAST(FECHA_HORA_DESEMBOLSO AS TIME) AS hora_desembolso,
        cast(TARGET_23_FG as date) as target_23_fg,
        CAST(TARGET_23_FG AS TIME) AS target_23_hg,
        DNI as vendor_lead_code,
        target_23_cg,
        target_23_cg,
        TARGET_23_OBS,
        tipo,
        desbase,
        monto_neto,
        monto_bruto,
        prestamo,
        canal_confirmado,
        perfil
        from VALENTINA.dbo.efectiva_ventas_desembolso
        where campana='{campana}'
        and FECHA_HORA_DESEMBOLSO >= DATEADD(MONTH, -4, DATEADD(DAY, 1, EOMONTH('{fecha_mes_base}'))) 
        and FECHA_HORA_DESEMBOLSO < DATEADD(MONTH, 0, DATEADD(DAY, 1, EOMONTH('{fecha_mes_base}')))
        """
    return pd.read_sql(query, engine_sa)





In [5]:
df_venta=since_sales(engine_zeus,fecha_mes_base,campana)

In [ ]:
df_venta=since_sales(engine_zeus,fecha_mes_base,campana)
